# Core-shell fluido: energia interna, mapas de estados ressonantes e relações de existência e de recorrência

Notebook ilustrativo do capítulo *Centro espalhador casca esférica fluida* da tese. Cada parte apresenta primeiro as
equações (na mesma notação do texto) e depois o código que as calcula e as figuras correspondentes. As relações de existência e de
recorrência são escritas em função do fator de mistura $f$, e um único ajuste vale para todo o intervalo $0\le f\le1$. Todos os
coeficientes são **ajustados aqui, a partir do banco de dados**; nenhum valor é copiado da tese.

1. Configurações e estética das figuras
2. Modelo: potenciais, condições de contorno e coeficientes
3. Energia interna no núcleo e na casca
4. Comparação com os casos-limite: casca fina, $\delta\to0$, e esfera fluida
5. Banco de dados e contagem das ressonâncias
6. Mapas de estados ressonantes e espectros
7. Relação de existência
8. Relação de recorrência
9. Tabela de coeficientes
10. Comparação com as formas ajustadas
11. Sistemas físicos: Liu *et al.* (2000) e Marston (2025)
12. Custo computacional
13. Arquivos gerados

As partes 1 a 4 só precisam deste notebook. As partes 5 a 12 leem a saída do gerador do core-shell (`gerador_coreshell.py`, versão 8):
uma pasta por fator de mistura em `coreshell_final/resultados_mix/` (`resumo_*.csv`, `ressonancias_*.csv`, `ressonancias_malha_*.csv` e
`checkpoint_*.npz`), mais a pasta das configurações sorteadas fora da grade, se houver.

## 1. Configurações
No Colab, a célula abaixo instala o LaTeX usado nos rótulos (só na primeira vez). No PC, basta ter uma distribuição LaTeX com `dvipng` e `cm-super`.

**Onde ficam os dados e os resultados.** No Colab, o Google Drive é montado uma única vez, logo no início. Os dados são lidos de
`Meu Drive/coreshell_final/resultados_mix/` e todas as figuras e tabelas são gravadas direto em `Meu Drive/coreshell_final/resultados/`,
de modo que nada se perde quando a sessão termina. Fora do Colab, a pasta dos dados vem da variável de ambiente `CORESHELL_DADOS`
(a pasta `resultados_mix` do gerador), e os resultados vão para `resultados/`, ao lado dela (ou para `CORESHELL_SAIDA`, se definida):

```
resultados/
├── figuras/   png (1200 dpi), pdf e eps de cada figura
└── tabelas/   csv, tex, png, pdf e svg de cada tabela, e relacoes_ajustadas.json
```

In [ ]:
%%capture
import os, sys, subprocess
if os.path.exists("/content"):
    subprocess.run("apt-get update -qq && apt-get install -y -qq texlive texlive-latex-extra texlive-fonts-recommended dvipng cm-super", shell=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])

In [ ]:
import os, re, glob, json, time, shutil, logging
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from matplotlib.colors import LinearSegmentedColormap, Normalize, LogNorm
from matplotlib.lines import Line2D
from matplotlib.patches import Rectangle, Circle
from matplotlib.ticker import FixedLocator, NullFormatter, NullLocator, FuncFormatter
from scipy.special import spherical_jn, spherical_yn, eval_legendre
from scipy.integrate import quad
from scipy.optimize import brentq, minimize_scalar
from scipy.signal import find_peaks
from scipy.spatial import cKDTree
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score

logging.getLogger("matplotlib.backends.backend_ps").setLevel(logging.ERROR)     # aviso de transparência no .eps (as figuras saem iguais)
USA_LATEX = shutil.which("latex") is not None
plt.rcParams.update({
    "text.usetex": USA_LATEX,
    "font.family": "serif",
    "font.serif": ["Computer Modern Roman"],
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "savefig.edgecolor": "white",
    "text.latex.preamble": r"\usepackage{amsmath}",
    "figure.dpi": 150,                      # só a exibição no notebook; os arquivos saem com OUTPUT_DPI
})
VIRG = "{,}" if USA_LATEX else ","          # vírgula decimal ("{,}" evita o espaço extra do LaTeX)
PCT = r"\%" if USA_LATEX else "%"

# estética padrão das figuras da tese
OUTPUT_DPI = 1200
COLORS = ["#6488A5", "#8B1A1A", "black", "0.45"]
LINESTYLES = ["-", "--", "-.", ":"]
LINEWIDTHS = [2.0, 2.0, 1.75, 1.6]

# ---------------- pastas: dados e resultados ----------------
# Google Drive: Meu Drive > coreshell_final (uma pasta por f em resultados_mix; resultados ao lado)
try:
    from google.colab import drive
    NO_COLAB = True
except ImportError:
    NO_COLAB = False

if NO_COLAB:
    drive.mount("/content/drive")
    PASTA = "/content/drive/MyDrive/coreshell_final/resultados_mix"          # saída do gerador
    SAIDA = "/content/drive/MyDrive/coreshell_final/resultados"              # figuras e tabelas, gravadas no Drive
else:
    if not os.environ.get("CORESHELL_DADOS"):
        raise RuntimeError("Fora do Colab, indique a pasta dos dados na variável de ambiente CORESHELL_DADOS (a pasta resultados_mix do gerador).")
    PASTA = os.path.abspath(os.environ["CORESHELL_DADOS"])
    SAIDA = os.path.abspath(os.environ.get("CORESHELL_SAIDA", os.path.join(os.path.dirname(PASTA), "resultados")))
FIG_DIR = os.path.join(SAIDA, "figuras")
TAB_DIR = os.path.join(SAIDA, "tabelas")
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TAB_DIR, exist_ok=True)


def salvar_figura(fig, nome):
    base = os.path.join(FIG_DIR, nome)
    fig.savefig(base + ".png", dpi=OUTPUT_DPI, bbox_inches="tight", pad_inches=0.02, facecolor="white", edgecolor="white")
    fig.savefig(base + ".pdf", bbox_inches="tight", pad_inches=0.02, facecolor="white", edgecolor="white")
    fig.savefig(base + ".eps", format="eps", bbox_inches="tight", pad_inches=0.02, facecolor="white", edgecolor="white")


def concluir(fig, nome):
    # grava, mostra e fecha a figura
    salvar_figura(fig, nome)
    plt.show()
    plt.close(fig)


def dec(valor, casas):
    return f"{valor:.{casas}f}".replace(".", VIRG)


def virgula(texto):
    # vírgula decimal num rótulo já escrito com ponto ("0.5" -> "0{,}5")
    return re.sub(r"(?<=\d)\.(?=\d)", VIRG, str(texto))


def estilizar(ax, xlabel=None, ylabel=None):
    if xlabel is not None:
        ax.set_xlabel(xlabel, fontsize=12)
    if ylabel is not None:
        ax.set_ylabel(ylabel, fontsize=12)
    ax.tick_params(axis="both", which="both", direction="in", length=4, width=0.9, labelsize=10)
    ax.tick_params(axis="both", which="minor", length=2)
    for spine in ax.spines.values():
        spine.set_linewidth(1.0)
        spine.set_edgecolor("black")
    ax.grid(False)


def adicionar_caixa(ax, texto, x=0.965, y=0.955, ha="right"):
    ax.text(x, y, texto, transform=ax.transAxes, ha=ha, va="top", fontsize=9, linespacing=1.10, zorder=10,
            bbox={"boxstyle": "square,pad=0.30", "facecolor": "white", "edgecolor": "black", "linewidth": 1.0, "alpha": 1.0})


def adicionar_legenda(ax, loc="lower right", **kw):
    leg = ax.legend(loc=loc, fontsize=kw.pop("fontsize", 9), frameon=True, handlelength=2.8, labelspacing=0.30,
                    handletextpad=0.7, **kw)
    leg.get_frame().set(edgecolor="black", facecolor="white", linewidth=1.0, alpha=1.0)
    return leg


def potencia(v, _=None):
    # rótulo de eixo log em potências de 10
    return "$1$" if np.isclose(v, 1) else rf"$10^{{{int(round(np.log10(v)))}}}$"


def gnum(v, sig=3):
    # número com vírgula decimal, em notação científica (mantissa x 10^e) fora de 0,01 <= |v| < 1000; sem os cifrões
    if not np.isfinite(v):
        return "--"
    if v == 0 or 1e-2 <= abs(v) < 1e3:
        return virgula(f"{v:.{sig}g}")
    e = int(np.floor(np.log10(abs(v))))
    return f"{dec(v / 10 ** e, sig - 1)}\\times10^{{{e}}}"


def milhar(n):
    return f"{int(n):,}".replace(",", r"\," if USA_LATEX else " ")


def eixo_log_y(ax, ticks, ylim):
    # eixo y logarítmico com ticks fixos em potências de 10 (mesmo padrão das curvas de energia da esfera)
    ax.set_yscale("log")
    ax.set_ylim(*ylim)
    ax.yaxis.set_major_locator(FixedLocator(ticks))
    ax.yaxis.set_major_formatter(FuncFormatter(potencia))
    ax.yaxis.set_minor_locator(NullLocator())
    ax.yaxis.set_minor_formatter(NullFormatter())


def formatar_log(ax, eixos="y"):
    # rótulos só nas potências de 10 de um eixo logarítmico
    for e in (ax.xaxis if "x" in eixos else None, ax.yaxis if "y" in eixos else None):
        if e is not None:
            e.set_major_formatter(FuncFormatter(potencia))
            e.set_minor_formatter(NullFormatter())


def eixo_x_fixo(ax, ticks, xlim=None):
    # ticks fixos no eixo x, com o mesmo número de casas e vírgula decimal
    casas = max((len(f"{float(t):.10f}".rstrip("0").split(".")[1]) for t in ticks), default=0)
    ax.set_xlim(*(xlim if xlim is not None else (ticks[0], ticks[-1])))
    ax.xaxis.set_major_locator(FixedLocator(ticks))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: "$0$" if v == 0 else f"${dec(v, casas)}$"))
    ax.xaxis.set_minor_locator(NullLocator())


# ---------------- estética dos mapas (plot_mapa_fase do notebook da esfera) ----------------
MAPA_RC = {"font.family": "sans-serif"}
BOX_COLOR = "#600B00"
CELL_LINECOLOR, CELL_LINEWIDTH = "white", 0.42
FRONT_OBS = dict(color="#7FA6D9", linestyle=(0, (5.0, 2.2)), linewidth=2.0, alpha=0.95)
FRONT_LOG = dict(color="#600B00", linestyle=(0, (4.5, 2.5)), linewidth=1.9, alpha=0.92)
RESONANCE_CMAP = LinearSegmentedColormap.from_list("resonance_count", ["white", "#D9E1EA", "#526F92", "black"], N=256)
RESONANCE_CMAP.set_bad("#D9D9D9")
LOG_Z = (-3.0, 3.0)                     # log10 de alpha e beta, de 1e-3 a 1e3
LARGURA_BIN = 0.30
NZ = int(round((LOG_Z[1] - LOG_Z[0]) / LARGURA_BIN))


def celula(v):
    # posição, em células, de um valor de alpha ou beta no mapa
    return (np.log10(v) - LOG_Z[0]) / LARGURA_BIN - 0.5


def figura_mapa(nx=NZ, ny=NZ):
    altura = 0.70
    largura = altura * nx / ny
    fig = plt.figure(figsize=(10.5, 10.5))
    return fig, fig.add_axes([0.12, 0.13, largura, altura]), fig.add_axes([0.12 + largura + 0.02, 0.13, 0.03, altura])


def desenhar_celulas(ax, matriz, norm, escala=lambda v: np.log10(1.0 + v)):
    for i in range(matriz.shape[0]):
        for j in range(matriz.shape[1]):
            v = matriz[i, j]
            cor = RESONANCE_CMAP(norm(escala(v))) if np.isfinite(v) else RESONANCE_CMAP.get_bad()
            ax.add_patch(Rectangle((i - 0.5, j - 0.5), 1.0, 1.0, facecolor=cor, edgecolor=CELL_LINECOLOR, linewidth=CELL_LINEWIDTH, zorder=1))


def segmentos_fronteira(mask):
    seg = []
    for i in range(mask.shape[0]):
        for j in range(mask.shape[1]):
            if not mask[i, j]:
                continue
            xl, xr, yb, yt = i - 0.5, i + 0.5, j - 0.5, j + 0.5
            if i == 0 or not mask[i - 1, j]: seg.append(((xl, yb), (xl, yt)))
            if i == mask.shape[0] - 1 or not mask[i + 1, j]: seg.append(((xr, yb), (xr, yt)))
            if j == 0 or not mask[i, j - 1]: seg.append(((xl, yb), (xr, yb)))
            if j == mask.shape[1] - 1 or not mask[i, j + 1]: seg.append(((xl, yt), (xr, yt)))
    return seg


def desenhar_fronteira(ax, mask, estilo, zorder=17):
    for a, b in segmentos_fronteira(mask):
        ax.plot([a[0], b[0]], [a[1], b[1]], solid_capstyle="butt", zorder=zorder, **estilo)


def caixa_mapa(ax, texto, x=0.955, y=0.965, ha="right"):
    ax.text(x, y, texto, transform=ax.transAxes, fontsize=24, color="white", va="top", ha=ha,
            bbox=dict(facecolor=BOX_COLOR, edgecolor="none", alpha=1.0, boxstyle="square,pad=0.14"), zorder=111)


def legenda_mapa(ax, handles, loc="upper right", bbox=(0.985, 0.85)):
    leg = ax.legend(handles=handles, loc=loc, bbox_to_anchor=bbox, bbox_transform=ax.transAxes, fontsize=20.0, frameon=True,
                    framealpha=1.0, facecolor="white", edgecolor="none", fancybox=False, borderpad=0.20, handlelength=2.4,
                    handletextpad=0.8, labelspacing=0.45, borderaxespad=0.0)
    leg.set_zorder(100)


def barra_mapa(fig, cax, norm, ticks, rotulos, rotulo):
    mapper = cm.ScalarMappable(norm=norm, cmap=RESONANCE_CMAP)
    mapper.set_array([])
    cbar = fig.colorbar(mapper, cax=cax, ticks=ticks)
    cbar.ax.set_yticklabels(rotulos)
    cbar.set_label(rotulo, fontsize=20, labelpad=18, rotation=270, color="black")
    cbar.minorticks_off()
    return cbar


def concluir_mapa(fig, ax, cbar, nome):
    for eixo in (ax, cbar.ax):
        eixo.tick_params(axis="both", labelsize=15, width=1.25, length=4.5, direction="out")
        for spine in eixo.spines.values():
            spine.set_visible(True)
            spine.set_edgecolor(BOX_COLOR)
            spine.set_linewidth(1.8)
    for ext in ("png", "pdf", "eps"):
        fig.savefig(os.path.join(FIG_DIR, f"{nome}.{ext}"), dpi=600, bbox_inches="tight")
    plt.show()
    plt.close(fig)


def ticks_ressonancias(n_max):
    # ticks da barra de cores em escala log(1 + N), como no mapa da esfera
    return np.unique(np.array([0] + [10 ** k for k in range(1, 6) if 10 ** k < 0.8 * n_max] + [n_max], dtype=int))


# ---------------- tabelas (save_compact_table do notebook da esfera) ----------------
def fmt_pm(valor, erro):
    if not (np.isfinite(erro) and erro > 0.0):
        return f"${dec(valor, 4)}$"
    casas = max(0, -int(np.floor(np.log10(erro))) + 1)
    return rf"${f'{valor:+.{casas}f}'.replace('.', VIRG)}\pm{f'{erro:.{casas}f}'.replace('.', VIRG)}$"


def save_compact_table(base, dados, linhas, colunas, figsize, first_col_width=0.16, header_label="Coef.", fontsize=17):
    with plt.rc_context({"font.family": "STIXGeneral", "mathtext.fontset": "stix", "font.size": 30}):
        fig, ax = plt.subplots(figsize=figsize, facecolor="white")
        ax.axis("off")
        tab = ax.table(cellText=[[linhas[i], *dados[i]] for i in range(len(linhas))], colLabels=[header_label, *colunas],
                       cellLoc="center", colWidths=[first_col_width] + [(1 - first_col_width) / len(colunas)] * len(colunas),
                       bbox=[0.02, 0.02, 0.96, 0.96])
        tab.auto_set_font_size(False)
        tab.set_fontsize(fontsize)
        for (lin, col), cel in tab.get_celld().items():
            cel.visible_edges = ""
            cel.set_linewidth(0.0)
            cel.PAD = 0.01
            cel.get_text().set_ha("left" if col == 0 else "center")
            cel.get_text().set_linespacing(0.90)
        for col in range(len(colunas) + 1):
            tab[(0, col)].visible_edges = "TB"
            tab[(0, col)].set_linewidth(1.5)
            tab[(0, col)].get_text().set_fontsize(fontsize + 1)
            tab[(len(linhas), col)].visible_edges = "B"
            tab[(len(linhas), col)].set_linewidth(1.5)
        tab[(0, 0)].get_text().set_fontweight("bold")
        fig.subplots_adjust(left=0.005, right=0.995, bottom=0.005, top=0.995)
        for ext in ("png", "pdf", "svg"):
            fig.savefig(f"{base}.{ext}", dpi=600, bbox_inches="tight", pad_inches=0.01, facecolor="white")
        plt.show()
        plt.close(fig)


def salvar_tabela(nome, linhas, colunas, dados, figsize, first_col_width=0.16, fontsize=17, header_label="Coef.", csv=None):
    # figura da tabela (png, pdf, svg) + .tex com as mesmas células + .csv com os números
    base = os.path.join(TAB_DIR, nome)
    save_compact_table(base, dados, linhas, colunas, figsize, first_col_width, header_label, fontsize)
    with open(base + ".tex", "w", encoding="utf-8") as fh:
        fh.write("\\begin{tabular}{l" + "c" * len(colunas) + "}\n\\hline\n")
        fh.write(header_label + " & " + " & ".join(c.replace("\n", " ") for c in colunas) + " \\\\\n\\hline\n")
        for i, l in enumerate(linhas):
            fh.write(l.replace("\n", " ") + " & " + " & ".join(d.replace("\n", " ") for d in dados[i]) + " \\\\\n")
        fh.write("\\hline\n\\end{tabular}\n")
    if csv is not None:
        csv.to_csv(base + ".csv", index=False)


ARQ_REL = os.path.join(TAB_DIR, "relacoes_ajustadas.json")


def gravar_relacoes(**novos):
    rel = {}
    if os.path.exists(ARQ_REL):
        with open(ARQ_REL, encoding="utf-8") as fh:
            rel = json.load(fh)
    rel.update(novos, data=time.strftime("%d/%m/%Y %H:%M"), pasta_dos_dados=PASTA)
    with open(ARQ_REL, "w", encoding="utf-8") as fh:
        json.dump(rel, fh, indent=1, ensure_ascii=False, default=float)


print("LaTeX:", USA_LATEX, "| Colab:", NO_COLAB, "\ndados:", PASTA, "\nfiguras:", FIG_DIR, "\ntabelas:", TAB_DIR)

## 2. Modelo

Uma onda plana monocromática, de potencial $\varphi_{\text{inc}}$, incide ao longo de $+z$ sobre uma partícula esférica formada por um núcleo
(região $A$, $0\le r<a$, densidade $\rho_A$ e velocidade do som $c_A$) envolvido por uma casca (região $B$, $a\le r\le b$, $\rho_B$ e $c_B$),
imersa em um fluido de densidade $\rho_M$ e velocidade do som $c_M$ (região $M$, $r>b$). A espessura relativa da casca é $\delta$, com $b=(1+\delta)a$.
Em cada região o potencial satisfaz a equação de Helmholtz, e

$$
\begin{aligned}
\varphi_{\text{inc}} &= A\sum_{\ell=0}^{\infty}(2\ell+1)\,i^\ell j_\ell(k_Mr)P_\ell(\cos\theta), &
\varphi_{A} &= A\sum_{\ell=0}^{\infty}(2\ell+1)\,i^\ell\, b_\ell\, j_\ell(k_Ar)P_\ell(\cos\theta),\\
\varphi_{B} &= A\sum_{\ell=0}^{\infty}(2\ell+1)\,i^\ell\big[c_\ell\, j_\ell(k_Br)+d_\ell\, y_\ell(k_Br)\big]P_\ell(\cos\theta), &
\varphi_{\text{sc}} &= A\sum_{\ell=0}^{\infty}(2\ell+1)\,i^\ell\, s_\ell\, h^{(1)}_\ell(k_Mr)P_\ell(\cos\theta).
\end{aligned}
$$

A continuidade da pressão e da velocidade normal em cada interface, $\rho_i\varphi_i=\rho_e\varphi_e$ e $\partial_r\varphi_i=\partial_r\varphi_e$,
gera, para cada $\ell$, um sistema linear $4\times4$ para $\mathbf x_\ell=[b_\ell,c_\ell,d_\ell,s_\ell]^T$:

$$
\begin{bmatrix}
0 & -j_\ell(y_B) & -y_\ell(y_B) & \rho_{MB}\,h^{(1)}_\ell(y_M)\\
0 & -j'_\ell(y_B) & -y'_\ell(y_B) & k_{MB}\,h^{(1)\prime}_\ell(y_M)\\
-\rho_{AB}\,j_\ell(x_A) & j_\ell(x_B) & y_\ell(x_B) & 0\\
-k_{AB}\,j'_\ell(x_A) & j'_\ell(x_B) & y'_\ell(x_B) & 0
\end{bmatrix}
\mathbf x_\ell=
\begin{bmatrix}-\rho_{MB}\,j_\ell(y_M)\\ -k_{MB}\,j'_\ell(y_M)\\ 0\\ 0\end{bmatrix},
$$

com $x_A=k_Aa$, $x_B=k_Ba$, $y_M=k_Mb$, $y_B=k_Bb$, $\rho_{ij}=\rho_i/\rho_j$ e $k_{ij}=k_i/k_j=c_j/c_i$. Tudo é escrito em função de
$x_M=k_Ma$: $x_A=(c_M/c_A)x_M$, $x_B=(c_M/c_B)x_M$, $y_M=(1+\delta)x_M$ e $y_B=(1+\delta)x_B$.

Com os wronskianos $j_\ell y_\ell'-j_\ell'y_\ell=1/s^2$ e $h^{(1)}_\ell j_\ell'-h^{(1)\prime}_\ell j_\ell=-i/s^2$, os coeficientes se escrevem com o elemento de contorno

$$
\Gamma^{fg}_{\mu\nu}=\frac{\rho_\mu}{\rho_\nu}\,f_\ell(s_\mu)\,g'_\ell(s_\nu)-\frac{c_\nu}{c_\mu}\,f'_\ell(s_\mu)\,g_\ell(s_\nu),
$$

e todos dividem o mesmo denominador $\mathcal D_\ell$:

$$
\begin{aligned}
b_\ell&=-\frac{i}{y_M^2}\frac{\rho_M}{\rho_B}\frac{c_B}{c_M}\frac{1}{\mathcal D_\ell\,x_B^2}, &
c_\ell&=-\frac{i}{y_M^2}\frac{\rho_M}{\rho_B}\frac{c_B}{c_M}\frac{\Gamma^{jy}_{AB}}{\mathcal D_\ell}, &
d_\ell&=\frac{i}{y_M^2}\frac{\rho_M}{\rho_B}\frac{c_B}{c_M}\frac{\Gamma^{jj}_{AB}}{\mathcal D_\ell},\\
s_\ell&=\frac{\Gamma^{jj}_{AB}\Gamma^{jy}_{MB}-\Gamma^{jy}_{AB}\Gamma^{jj}_{MB}}{\mathcal D_\ell}, &
\mathcal D_\ell&=\Gamma^{jy}_{AB}\Gamma^{hj}_{MB}-\Gamma^{jj}_{AB}\Gamma^{hy}_{MB}.
\end{aligned}
$$

Os zeros de $\mathcal D_\ell$ ficam no plano complexo, em $x_M=x_k-i\gamma_k$; quando um deles está perto do eixo real, os coeficientes crescem em torno de
$x_M=x_k$, com largura $2\gamma_k$, e aparece uma ressonância.

**Parâmetros.** As impedâncias são escritas em relação ao meio, $\alpha=Z_A/Z_M$ e $\beta=Z_B/Z_M$, e o fator de mistura $f$ reparte o contraste entre
densidade e velocidade:

$$
\frac{\rho_A}{\rho_M}=\alpha^f,\qquad \frac{c_A}{c_M}=\alpha^{1-f},\qquad \frac{\rho_B}{\rho_M}=\beta^f,\qquad \frac{c_B}{c_M}=\beta^{1-f},
$$

de modo que $f=0$ põe todo o contraste na velocidade e $f=1$, todo na densidade. A assimetria entre núcleo e casca é $\eta=(\alpha-\beta)/(\alpha+\beta)$.
As funções de ordem negativa seguem $j_{-n-1}=(-1)^{n+1}y_n$, de modo que $j_{-1}(x)=\cos x/x$.

A célula seguinte confere as fórmulas fechadas contra a solução numérica do sistema $4\times4$.

In [ ]:
LMAX = 10
X_MIN, X_MAX = 1e-4, 1.0
RHO_AGUA, C_AGUA = 1000.0, 1480.0          # meio externo das grades do gerador


def J(n, x):
    if n >= 0:
        return spherical_jn(n, x)
    return (-1) ** (-n) * spherical_yn(-n - 1, x)


def Y(n, x):
    if n >= 0:
        return spherical_yn(n, x)
    return (-1) ** (-n - 1) * spherical_jn(-n - 1, x)


def dJ(n, x):
    # f_l' = f_{l-1} - (l+1)/x f_l (vale para qualquer ordem)
    return J(n - 1, x) - (n + 1) / x * J(n, x)


def dY(n, x):
    return Y(n - 1, x) - (n + 1) / x * Y(n, x)


def H(n, x):
    return J(n, x) + 1j * Y(n, x)


def dH(n, x):
    return dJ(n, x) + 1j * dY(n, x)


def materiais(alpha, beta, f):
    # (rho_M, rho_A, rho_B, c_M, c_A, c_B) com rho_i/rho_M = (Z_i/Z_M)^f e c_i/c_M = (Z_i/Z_M)^(1-f), meio = água da grade
    return RHO_AGUA, RHO_AGUA * alpha ** f, RHO_AGUA * beta ** f, C_AGUA, C_AGUA * alpha ** (1 - f), C_AGUA * beta ** (1 - f)


def argumentos(xM, delta, mat):
    rM, rA, rB, cM, cA, cB = mat
    xM = np.asarray(xM)
    xB = (cM / cB) * xM
    return (cM / cA) * xM, xB, (1 + delta) * xM, (1 + delta) * xB


def gamma_contorno(f, df, g, dg, rho_mu_nu, c_nu_mu):
    # Gamma^{fg}_{mu nu} = (rho_mu/rho_nu) f(s_mu) g'(s_nu) - (c_nu/c_mu) f'(s_mu) g(s_nu)
    return rho_mu_nu * f * dg - c_nu_mu * df * g


def coeficientes(ell, xM, delta, mat):
    # b_l, c_l, d_l, s_l e D_l (vale também para x_M complexo)
    rM, rA, rB, cM, cA, cB = mat
    xA, xB, yM, yB = argumentos(xM, delta, mat)
    with np.errstate(all="ignore"):
        jA, djA = J(ell, xA), dJ(ell, xA)
        jB, djB, yxB, dyxB = J(ell, xB), dJ(ell, xB), Y(ell, xB), dY(ell, xB)
        jb, djb, yb, dyb = J(ell, yB), dJ(ell, yB), Y(ell, yB), dY(ell, yB)
        jM, djM, hM, dhM = J(ell, yM), dJ(ell, yM), H(ell, yM), dH(ell, yM)
        G_jj_AB = gamma_contorno(jA, djA, jB, djB, rA / rB, cB / cA)
        G_jy_AB = gamma_contorno(jA, djA, yxB, dyxB, rA / rB, cB / cA)
        G_hj_MB = gamma_contorno(hM, dhM, jb, djb, rM / rB, cB / cM)
        G_hy_MB = gamma_contorno(hM, dhM, yb, dyb, rM / rB, cB / cM)
        G_jj_MB = gamma_contorno(jM, djM, jb, djb, rM / rB, cB / cM)
        G_jy_MB = gamma_contorno(jM, djM, yb, dyb, rM / rB, cB / cM)
        D = G_jy_AB * G_hj_MB - G_jj_AB * G_hy_MB
        pre = -1j / yM ** 2 * (rM / rB) * (cB / cM)
        return pre / (D * xB ** 2), pre * G_jy_AB / D, -pre * G_jj_AB / D, (G_jj_AB * G_jy_MB - G_jy_AB * G_jj_MB) / D, D


def coeficientes_sistema(ell, xM, delta, mat):
    # solução numérica do sistema 4 x 4, só para conferência
    rM, rA, rB, cM, cA, cB = mat
    xA, xB, yM, yB = argumentos(xM, delta, mat)
    rMB, rAB, kMB, kAB = rM / rB, rA / rB, cB / cM, cB / cA
    Mx = np.array([[0, -J(ell, yB), -Y(ell, yB), rMB * H(ell, yM)], [0, -dJ(ell, yB), -dY(ell, yB), kMB * dH(ell, yM)],
                   [-rAB * J(ell, xA), J(ell, xB), Y(ell, xB), 0], [-kAB * dJ(ell, xA), dJ(ell, xB), dY(ell, xB), 0]], dtype=complex)
    return np.linalg.solve(Mx, np.array([-rMB * J(ell, yM), -kMB * dJ(ell, yM), 0, 0], dtype=complex))


pior = 0.0
for (alpha, beta, delta, f) in [(0.01, 0.003, 0.5, 0.0), (9.65, 0.0099, 0.42, 0.5), (0.2, 30.0, 1.35, 1.0), (1000.0, 0.05, 0.1, 0.25)]:
    mat = materiais(alpha, beta, f)
    for ell in (0, 1, 3, 6):
        for xM in (0.003, 0.07, 0.6):
            fechada = np.array(coeficientes(ell, xM, delta, mat)[:4])
            pior = max(pior, np.max(np.abs(fechada - coeficientes_sistema(ell, xM, delta, mat)) / np.abs(fechada)))
print(f"maior diferença relativa entre as fórmulas fechadas e o sistema 4 x 4: {pior:.1e}")

## 3. Energia interna

Em um fluido ideal, a energia acústica média em um volume $V$ é

$$
W_T=\frac{\rho}{4}\int_V|\nabla\varphi|^2\,dV+\frac{\rho k^2}{4}\int_V|\varphi|^2\,dV .
$$

**Núcleo.** O potencial tem a mesma forma da esfera homogênea, com o coeficiente $b_\ell$ carregando a influência da casca:

$$
\frac{2f_0W_P^{(A)}}{Z_A|A|^2}=\frac{x_A^3}{2}\sum_{\ell=0}^{\infty}|b_\ell|^2(2\ell+1)\Big[j_\ell^2(x_A)-j_{\ell-1}(x_A)j_{\ell+1}(x_A)\Big],
\qquad
\frac{2f_0W_K^{(A)}}{Z_A|A|^2}=\sum_{\ell=0}^{\infty}|b_\ell|^2(2\ell+1)\left\{\frac{x_A^3}{2}\Big[j_\ell^2-j_{\ell-1}j_{\ell+1}\Big]+x_A^2\,j_\ell\,j_\ell'\right\}_{x_A}.
$$

**Casca.** Com as primitivas

$$
F_{jj}=\Big[x^3\big(j_\ell^2-j_{\ell-1}j_{\ell+1}\big)\Big]_{x_B}^{y_B},\quad
F_{yy}=\Big[x^3\big(y_\ell^2-y_{\ell-1}y_{\ell+1}\big)\Big]_{x_B}^{y_B},\quad
F_{jy}=\Big[\frac{x^3}{2}\big(2j_\ell y_\ell-j_{\ell+1}y_{\ell-1}-j_{\ell-1}y_{\ell+1}\big)\Big]_{x_B}^{y_B},
$$

a energia potencial é

$$
\frac{2f_0W_P^{(B)}}{Z_B|A|^2}=\sum_{\ell=0}^{\infty}(2\ell+1)\left[\frac{|c_\ell|^2}{2}F_{jj}+\frac{|d_\ell|^2}{2}F_{yy}+\mathrm{Re}(c_\ell^*d_\ell)F_{jy}\right],
$$

e, depois das integrais radiais por partes e da parcela angular, a cinética é a potencial mais termos de superfície nas duas fronteiras:

$$
\frac{2f_0W_K^{(B)}}{Z_B|A|^2}=\frac{2f_0W_P^{(B)}}{Z_B|A|^2}+\sum_{\ell=0}^{\infty}(2\ell+1)\Big\{|c_\ell|^2\big[x^2j_\ell j_\ell'\big]_{x_B}^{y_B}
+|d_\ell|^2\big[x^2y_\ell y_\ell'\big]_{x_B}^{y_B}+2\,\mathrm{Re}(c_\ell^*d_\ell)\big[x^2j_\ell y_\ell'\big]_{x_B}^{y_B}\Big\}.
$$

**Energia total.** Cada região sai normalizada pela própria impedância, $\widetilde W_A=2f_0W_T^{(A)}/(Z_A|A|^2)$ e
$\widetilde W_B=2f_0W_T^{(B)}/(Z_B|A|^2)$; para somar com a mesma referência ($Z_M$), como no gerador,

$$
\widetilde W_T=\frac{2f_0\,(W_T^{(A)}+W_T^{(B)})}{Z_M|A|^2}=\alpha\,\widetilde W_A+\beta\,\widetilde W_B .
$$

Só a soma em $\ell$ precisa ser truncada; adota-se $\ell_{\max}=10$, o mesmo do banco de dados. A célula seguinte confere as primitivas contra a
integração numérica: $\int x^2j_\ell^2\,dx=F_{jj}/2$, $\int x^2 y_\ell^2\,dx=F_{yy}/2$, $\int x^2j_\ell y_\ell\,dx=F_{jy}/2$, e a parte radial com a angular,
$\int\big[x^2j_\ell'^2+\ell(\ell+1)j_\ell^2\big]dx=\big[x^2j_\ell j_\ell'\big]+F_{jj}/2$.

In [ ]:
def prim_F(ell, x, f, g):
    # x^3 [f_l g_l - (f_{l-1} g_{l+1} + f_{l+1} g_{l-1})/2]: com f = g = j dá x^3 (j_l^2 - j_{l-1} j_{l+1})
    return x ** 3 * (f(ell, x) * g(ell, x) - 0.5 * (f(ell - 1, x) * g(ell + 1, x) + f(ell + 1, x) * g(ell - 1, x)))


def prim_B(ell, x, f, g):
    # termo de superfície x^2 f_l g_l'
    return x ** 2 * f(ell, x) * (g(ell - 1, x) - (ell + 1) / x * g(ell, x))


def energia_modal(ell, xM, delta, mat):
    # contribuição do multipolo l: (W~_T, W~_A, W~_B); W~_A e W~_B normalizadas por Z_A e Z_B, W~_T = alpha W~_A + beta W~_B (por Z_M)
    rM, rA, rB, cM, cA, cB = mat
    xA, xB, yM, yB = argumentos(np.asarray(xM, float), delta, mat)
    with np.errstate(all="ignore"):
        b, c, d, _, _ = coeficientes(ell, xM, delta, mat)
        p = 2 * ell + 1
        WA = np.real(np.abs(b) ** 2 * p * (prim_F(ell, xA, J, J) + prim_B(ell, xA, J, J)))                 # W_P + W_K do núcleo
        Fjj, Fyy, Fjy = (prim_F(ell, yB, f, g) - prim_F(ell, xB, f, g) for f, g in ((J, J), (Y, Y), (J, Y)))
        Bjj, Byy, Bjy = (prim_B(ell, yB, f, g) - prim_B(ell, xB, f, g) for f, g in ((J, J), (Y, Y), (J, Y)))
        c2, d2, cd = np.abs(c) ** 2, np.abs(d) ** 2, np.real(np.conj(c) * d)
        WP = p * (0.5 * c2 * Fjj + 0.5 * d2 * Fyy + cd * Fjy)
        WB = np.real(2 * WP + p * (c2 * Bjj + d2 * Byy + 2 * cd * Bjy))                                     # W_P + W_K da casca
        WT = rA * cA / (rM * cM) * WA + rB * cB / (rM * cM) * WB
    return [np.where(np.isfinite(w), w, np.nan) for w in (WT, WA, WB)]


def energia_total(xM, delta, mat, lmax=LMAX, bloco=200_000):
    # W~_T = soma de l = 0 a lmax (em blocos, para malhas de milhões de pontos)
    xM = np.asarray(xM, float)
    if len(xM) > bloco:
        return np.concatenate([energia_total(xM[i:i + bloco], delta, mat, lmax) for i in range(0, len(xM), bloco)])
    W = np.zeros_like(xM)
    for ell in range(lmax + 1):
        W += np.nan_to_num(energia_modal(ell, xM, delta, mat)[0])
    return np.where(np.isfinite(W) & (W > 0), W, 0.0)


def energias_regioes(xM, delta, mat, lmax=LMAX):
    # (W~_A, W~_B): energia do núcleo (por Z_A) e da casca (por Z_B), somadas em l
    WA, WB = 0.0, 0.0
    for ell in range(lmax + 1):
        _, a, b = energia_modal(ell, xM, delta, mat)
        WA, WB = WA + np.nan_to_num(a), WB + np.nan_to_num(b)
    return WA, WB


print(" l     x_B     y_B    integral j^2 x^2   F_jj/2       integral y^2 x^2   F_yy/2       integral (cinética)   [x^2 j j'] + F_jj/2")
for ell in (0, 1, 3):
    for xB, yB in ((0.4, 1.0), (3.0, 7.5)):
        nJ = quad(lambda t: t ** 2 * J(ell, t) ** 2, xB, yB, limit=200)[0]
        nY = quad(lambda t: t ** 2 * Y(ell, t) ** 2, xB, yB, limit=200)[0]
        nK = quad(lambda t: t ** 2 * dJ(ell, t) ** 2 + ell * (ell + 1) * J(ell, t) ** 2, xB, yB, limit=200)[0]
        Fjj = prim_F(ell, yB, J, J) - prim_F(ell, xB, J, J)
        Fyy = prim_F(ell, yB, Y, Y) - prim_F(ell, xB, Y, Y)
        Bjj = prim_B(ell, yB, J, J) - prim_B(ell, xB, J, J)
        print(f"{ell:2d} {xB:7.2f} {yB:7.2f} {nJ:16.9e} {Fjj / 2:12.9e} {nY:18.9e} {Fyy / 2:12.9e} {nK:18.9e} {Bjj + Fjj / 2:20.9e}")

## 4. Comparação com os casos-limite

A formulação do core-shell precisa recuperar a esfera fluida homogênea (capítulo anterior) quando a distinção entre as regiões desaparece.
Para uma esfera de material $S$ e raio $R$ em $M$, com $x=k_MR$, $x_S=(c_M/c_S)x$ e $m_t=Z_M/Z_S$,

$$
b^{\text{esf}}_\ell=\frac{i\,x^{-2}\,(\rho_M/\rho_S)}{D^{\text{esf}}_\ell},\qquad
s^{\text{esf}}_\ell=-\frac{j_\ell'(x)j_\ell(x_S)-m_t\,j_\ell(x)j_\ell'(x_S)}{D^{\text{esf}}_\ell},\qquad
D^{\text{esf}}_\ell=h_\ell^{(1)\prime}(x)\,j_\ell(x_S)-m_t\,j_\ell'(x_S)\,h_\ell^{(1)}(x),
$$

e a energia interna é $\dfrac{2f_0W_T^{\text{esf}}}{Z_S|A|^2}=\sum_\ell|b^{\text{esf}}_\ell|^2(2\ell+1)\big[2F_\ell(x_S)+x_S^2j_\ell(x_S)j_\ell'(x_S)\big]$, com
$F_\ell(x_S)=\tfrac{x_S^3}{2}\big[j_\ell^2(x_S)-j_{\ell-1}(x_S)j_{\ell+1}(x_S)\big]$. Os três limites são:

1. **Casca fina** ($\delta\to0$, $b\to a$): a partícula é o núcleo $A$ de raio $a$; $b_\ell\to b^{\text{esf}}_\ell$, $s_\ell\to s^{\text{esf}}_\ell$, a casca fica sem
volume, a energia nela vai a zero e $\widetilde W_T\to\alpha\,\widetilde W^{\text{esf}}_A$.
2. **Núcleo e casca iguais** ($A=B$): o campo da casca é a continuação do campo do núcleo, $c_\ell=b_\ell$, $d_\ell=0$, e $s_\ell$ e a energia total
são os da esfera de raio externo $b=(1+\delta)a$; em $k_Mb$, o espaçamento das ressonâncias volta a $\pi/m$.
3. **Casca igual ao meio** ($B=M$): a interface externa some, $\mathcal D_\ell\propto j_\ell(x_A)\,h^{(1)\prime}_\ell(x_M)-\frac{Z_M}{Z_A}\,j_\ell'(x_A)\,h^{(1)}_\ell(x_M)$,
$s_\ell\to s^{\text{esf}}_\ell(x_M)$ e a energia do núcleo é a da esfera $A$ de raio $a$.

A célula seguinte confere os três limites nos coeficientes **e na energia**, mostrando numericamente que as diferenças vão a zero; a figura
seguinte compara as energias no limite homogêneo com a casca fina.

In [ ]:
def esfera(ell, x, mat_esfera):
    # coeficientes da esfera homogênea S de raio R em M, com x = k_M R; mat_esfera = (rho_M, rho_S, c_M, c_S)
    rM, rS, cM, cS = mat_esfera
    xS, mt = (cM / cS) * x, (rM * cM) / (rS * cS)
    with np.errstate(all="ignore"):
        D = dH(ell, x) * J(ell, xS) - mt * dJ(ell, xS) * H(ell, x)
        return 1j / x ** 2 * (rM / rS) / D, -(dJ(ell, x) * J(ell, xS) - mt * J(ell, x) * dJ(ell, xS)) / D


def energia_esfera(x, mat_esfera, lmax=LMAX):
    # W~ da esfera homogênea normalizada pela própria impedância Z_S
    rM, rS, cM, cS = mat_esfera
    xS, W = (cM / cS) * np.asarray(x, float), 0.0
    for ell in range(lmax + 1):
        b, _ = esfera(ell, x, mat_esfera)
        W = W + np.real(np.abs(b) ** 2 * (2 * ell + 1) * (prim_F(ell, xS, J, J) + prim_B(ell, xS, J, J)))
    return W


def dif_rel(a, b):
    return float(np.max(np.abs(a - b) / np.abs(b)))


rM, cM = 1024.0, 1522.0                      # água do mar
rA, cA, rB, cB = 1.225, 343.0, 1200.0, 1700.0  # núcleo de ar e casca qualquer
x = np.array([0.004, 0.05, 0.3, 0.8])
xg = np.geomspace(1e-3, 0.9, 400)
print("1. casca fina, delta -> 0, núcleo de ar: comparação com a esfera A de raio a")
for d in (1e-3, 1e-5, 1e-7):
    print(f"   delta = {d:.0e}: " + "   ".join(f"l = {ell}: |b - b_esf|/|b| = {dif_rel(coeficientes(ell, x, d, (rM, rA, rB, cM, cA, cB))[0], esfera(ell, x, (rM, rA, cM, cA))[0]):.1e}"
                                      for ell in (0, 1, 4)))
    WA, WB = energias_regioes(xg, d, (rM, rA, rB, cM, cA, cB))
    print(f"      energia: |W~_A - W~_esf|/W~_esf = {dif_rel(WA, energia_esfera(xg, (rM, rA, cM, cA))):.1e} | W~_B/W~_A máximo = {np.max(WB / WA):.1e}")
print("2. A = B (delta = 0,6): c = b, d = 0, s e energia iguais aos da esfera de raio b = (1 + delta) a")
for ell in (0, 1, 4):
    b, c, d, s, _ = coeficientes(ell, x, 0.6, (rM, rB, rB, cM, cB, cB))
    print(f"   l = {ell}: |c - b|/|b| = {dif_rel(c, b):.1e}   |d|/|b| = {np.max(np.abs(d) / np.abs(b)):.1e}   "
          f"|s - s_esf|/|s| = {dif_rel(s, esfera(ell, 1.6 * x, (rM, rB, cM, cB))[1]):.1e}")
WA, WB = energias_regioes(xg, 0.6, (rM, rB, rB, cM, cB, cB))
print(f"   energia: |(W~_A + W~_B) - W~_esf|/W~_esf = {dif_rel(WA + WB, energia_esfera(1.6 * xg, (rM, rB, cM, cB))):.1e}")
print("3. B = M (delta = 0,6): s e energia do núcleo iguais aos da esfera A de raio a")
for ell in (0, 1, 4):
    print(f"   l = {ell}: |s - s_esf|/|s| = {dif_rel(coeficientes(ell, x, 0.6, (rM, rA, rM, cM, cA, cM))[3], esfera(ell, x, (rM, rA, cM, cA))[1]):.1e}")
WA, _ = energias_regioes(xg, 0.6, (rM, rA, rM, cM, cA, cM))
print(f"   energia do núcleo: |W~_A - W~_esf|/W~_esf = {dif_rel(WA, energia_esfera(xg, (rM, rA, cM, cA))):.1e}")

### Limite homogêneo

Núcleo e casca com o mesmo material, contraste só na velocidade ($c_A/c_M=c_B/c_M=0{,}01$, água do mar como meio externo) e casca fina
($\delta=0{,}01$): a energia total do core-shell coincide com a da esfera homogênea de raio $b$, e cada região guarda a sua parcela.

In [ ]:
def figura_limite_homogeneo(xM, alpha=0.01, delta=0.01, rho0=1024.0, c0=1522.0):
    # limite homogêneo com contraste na velocidade: rho_A = rho_B = rho_M e c_A = c_B = alpha c_M
    mat = (rho0, rho0, rho0, c0, alpha * c0, alpha * c0)
    WA, WB = energias_regioes(xM, delta, mat)
    W_esf = energia_esfera((1 + delta) * xM, (rho0, rho0, c0, alpha * c0))
    d = virgula(f"{delta:g}")
    print(f"maior diferença relativa entre a energia total do core-shell e a da esfera de raio b: {dif_rel(WA + WB, W_esf):.1e}")
    fig, ax = plt.subplots(figsize=(5, 3.5))
    for i, (W, rot) in enumerate(((W_esf, r"Esfera homogênea ($\delta\to0$)"), (WA + WB, r"Energia total (casca-núcleo, $\delta\to0$)"),
                                  (WA, rf"Energia do núcleo ($\delta={d}$)"), (WB, rf"Energia da casca ($\delta={d}$)"))):
        ok = np.isfinite(W) & (W > 0)
        ax.plot(xM[ok], W[ok], color=COLORS[i], linestyle=LINESTYLES[i], linewidth=LINEWIDTHS[i], label=rot, zorder=2 + i)
    eixo_x_fixo(ax, [0.0, 0.025, 0.05, 0.075, 0.1])
    eixo_log_y(ax, [1e-8, 1e-4, 1e0, 1e4, 1e8], (1e-8, 1e9))
    estilizar(ax, r"$x_M$", r"Energia interna $(u.a.)$")
    adicionar_legenda(ax, fontsize=8)
    fig.subplots_adjust(left=0.16, right=0.985, bottom=0.15, top=0.98)
    concluir(fig, "limite_homogeneo")


figura_limite_homogeneo(np.linspace(1e-4, 0.1, 1000))

## 5. Banco de dados e contagem das ressonâncias

O gerador calcula $W_T$ para $13$ valores do fator de mistura, $f=0$; $0{,}10$; $0{,}20$; $0{,}25$; $0{,}30$; $0{,}40$; $0{,}50$; $0{,}60$; $0{,}70$; $0{,}75$; $0{,}80$; $0{,}90$ e $1$.
Para cada $f$ são $6\,800$ curvas: $17$ espessuras, de $\delta=0{,}01$ a $1{,}5$, e $400$ pares $(\alpha,\beta)$ por espessura, com
$10^{-3}\le\alpha,\beta\le3\times10^3$, escolhidos de modo a distribuir as configurações de forma balanceada em $\eta$. Somam-se a elas configurações
sorteadas fora da grade, com $\alpha$ e $\beta$ uniformes em escala log, $\delta$ uniforme entre $0{,}01$ e $1{,}5$ e $f$ uniforme entre $0$ e $1$.
Grades e configurações sorteadas formam um único banco, e cada configuração entra nos ajustes com o seu próprio $f$.

**Contagem.** Quando $\alpha\ge1$ e $\beta\ge1$, os picos de $W_T$ são largos e ficam resolvidos na malha base de $5\,000$ pontos ($1\,000$ logarítmicos em
$[10^{-4};0{,}1]$, $3\,000$ lineares em $[0{,}1;0{,}9]$ e $1\,000$ lineares em $[0{,}9;1]$); cada máximo com proeminência de pelo menos $0{,}1$ em $\log_{10}W_T$ é uma
ressonância. Quando $\alpha<1$ ou $\beta<1$, a malha é refinada em torno de cada polo, com passo definido pelo espaçamento $\Delta_W$, e um polo conta
quando tem fator de qualidade $Q\ge2$ e área de pelo menos $0{,}1\,\%$ da energia numa janela de largura $\Delta_W$
(`ressonancias_*.csv`, contagem final `N_res_energy`). Os picos medidos na malha ficam em `ressonancias_malha_*.csv`; em ambos, `primary_l` é a ordem
$\ell$ do polo associado a cada ressonância, e é com ela que se formam as famílias da relação de recorrência.

**Pastas.** A busca percorre todas as subpastas de `PASTA` (pula só o que estiver dentro da pasta de saída, `resultados/`; uma pasta como
`resultados_mix/` não é pulada). Para cada $f$ fica a pasta mais recente entre as que têm a grade completa (pelo menos $90\,\%$ das curvas da maior);
uma pasta cujo resumo tem $f$ variando entre as configurações é o conjunto sorteado fora da grade.

In [ ]:
def curva_id(df, nome):
    # identificador da curva: arquivo + idd_ip do gerador (ou os próprios parâmetros)
    if "idd" in df:
        chave = df["idd"].astype(str) + ("_" + df["ip"].astype(str) if "ip" in df else "")
    else:
        chave = df[[c for c in ("alpha", "beta", "delta", "mix_factor") if c in df]].round(10).astype(str).agg("_".join, axis=1)
    return nome + ":" + chave


def localizar_dados(pasta):
    # devolve as grades (uma pasta por f) e as pastas fora da grade; não pula pastas que só começam com o nome da saída
    grades, fora, candidatas = {}, [], {}
    saida = os.path.abspath(SAIDA) + os.sep
    for arq in sorted(glob.glob(os.path.join(pasta, "**", "resumo_*.csv"), recursive=True)):
        if os.path.abspath(arq).startswith(saida):
            continue
        dirn, nome = os.path.dirname(arq), os.path.basename(arq)[len("resumo_"):-4]
        item = {"pasta": dirn, "nome": nome, "resumo": arq, "mtime": os.path.getmtime(arq)}
        for k, prefixo, ext in (("res", "ressonancias_", ".csv"), ("malha", "ressonancias_malha_", ".csv"), ("ckpt", "checkpoint_", ".npz")):
            caminho = os.path.join(dirn, prefixo + nome + ext)
            item[k] = caminho if os.path.exists(caminho) else None
        cab = pd.read_csv(arq, nrows=5000)
        fs = np.unique(np.round(cab["mix_factor"].dropna().to_numpy(float), 4)) if "mix_factor" in cab else np.array([])
        m = re.search(r"mix(\d+\.\d+)", nome)
        if len(fs) > 1 or re.search(r"fora|aleat|sorte|random", os.path.relpath(arq, pasta), re.I):
            fora.append(item)
            continue
        f = float(m.group(1)) if m else 0.0 if "velocity" in nome else 1.0 if "density" in nome else (float(fs[0]) if len(fs) else np.nan)
        if np.isfinite(f):
            item["linhas"] = sum(1 for _ in open(arq, encoding="utf-8", errors="ignore")) - 1
            candidatas.setdefault(f, []).append(item)
    for f, cs in sorted(candidatas.items()):
        nmax = max(c["linhas"] for c in cs)
        grades[f] = max((c for c in cs if c["linhas"] >= 0.9 * nmax), key=lambda c: c["mtime"])
        for c in cs:
            if c is not grades[f]:
                print(f"(ignorada para f = {f:.2f}: {c['pasta']})")
    if not grades and not fora:
        sub = sorted(d for d in os.listdir(pasta) if os.path.isdir(os.path.join(pasta, d))) if os.path.isdir(pasta) else []
        raise FileNotFoundError(f"Nenhum resumo_*.csv encontrado.\nProcurado em: {pasta} (e em todas as subpastas)\n"
                                + (f"Subpastas existentes: {sub}" if os.path.isdir(pasta) else "A pasta não existe."))
    return grades, fora


def conjuntos():
    # (rótulo, f fixo da pasta ou nan, item) de todas as pastas lidas
    return [("grade", f, a) for f, a in GRADES.items()] + [("fora", np.nan, a) for a in FORA]


def ler_resumos():
    partes = []
    for conjunto, f, a in conjuntos():
        df = pd.read_csv(a["resumo"])
        df = df.loc[df["status"] == "ok"].copy()
        df["f"] = df["mix_factor"].astype(float) if np.isnan(f) else f
        df["N"] = (df["N_res_energy"] if "N_res_energy" in df else df["has_resonance"]).astype(float)
        df["curva"], df["conjunto"] = curva_id(df, a["nome"]), conjunto
        cols = ["curva", "conjunto", "alpha", "beta", "delta", "f", "N"] + [c for c in ("runtime_s", "curva_risco") if c in df]
        partes.append(df[cols])
    r = pd.concat(partes, ignore_index=True)
    mat = materiais(r["alpha"].to_numpy(), r["beta"].to_numpy(), r["f"].to_numpy())
    r["eta"] = (r["alpha"] - r["beta"]) / (r["alpha"] + r["beta"])
    r["Phi"] = mat[3] / mat[4] + r["delta"] * mat[3] / mat[5]                    # fase acumulada por unidade de x_M
    r["ressonante"] = (r["N"] > 0).astype(int)
    return r


def ler_ressonancias(chave):
    partes = []
    cols = ["idd", "ip", "alpha", "beta", "delta", "mix_factor", "xM_peak", "primary_l", "FWHM_energy", "Q_energy", "WT_peak", "origem"]
    for conjunto, f, a in conjuntos():
        if a[chave] is None or os.path.getsize(a[chave]) < 3:
            continue
        df = pd.read_csv(a[chave], usecols=lambda c: c in cols)
        for c in ("primary_l", "WT_peak", "FWHM_energy", "Q_energy", "origem"):
            if c not in df:
                df[c] = np.nan
        df["f"] = df["mix_factor"].astype(float) if np.isnan(f) else f
        df["curva"], df["conjunto"] = curva_id(df, a["nome"]), conjunto
        partes.append(df.drop(columns=[c for c in ("idd", "ip", "mix_factor") if c in df]))
    if not partes:
        return pd.DataFrame(columns=["curva", "conjunto", "alpha", "beta", "delta", "f", "xM_peak", "primary_l", "WT_peak", "origem"])
    df = pd.concat(partes, ignore_index=True)
    return df.loc[np.isfinite(df["xM_peak"]) & (df["xM_peak"] >= X_MIN) & (df["xM_peak"] <= X_MAX)].reset_index(drop=True)


GRADES, FORA = localizar_dados(PASTA)
RESUMO = ler_resumos()
RES = ler_ressonancias("res")              # contagem final (polos com Q >= 2 e área, ou malha)
MALHA = ler_ressonancias("malha")          # máximos de W_T na malha
RESUMO["N_malha"] = RESUMO["curva"].map(MALHA.groupby("curva").size()).fillna(0).astype(int)
print(f"pasta: {PASTA}\ngrades: {len(GRADES)} valores de f | configurações fora da grade: {int((RESUMO['conjunto'] == 'fora').sum())}")
print(f"\n   f     curvas   ressonantes   N máximo   picos (contagem final)" + ("   tempo mediano do gerador (s)" if "runtime_s" in RESUMO else ""))
for f, d in RESUMO.loc[RESUMO["conjunto"] == "grade"].groupby("f"):
    t = f"   {d['runtime_s'].median():10.2f}" if "runtime_s" in d else ""
    print(f"{f:5.2f} {len(d):9d} {100 * d['ressonante'].mean():10.1f} % {int(d['N'].max()):9d} {int(d['N'].sum()):14d}{t}")
print(f"\nressonâncias com l associado: {100 * RES['primary_l'].notna().mean():.1f} % (contagem final) | "
      f"{100 * MALHA['primary_l'].notna().mean():.1f} % (picos da malha)")

### Conferência da energia e picos espúrios

**Energia.** Nos picos bem resolvidos (curvas com $\alpha\ge1$ e $\beta\ge1$, contados na malha base), a altura gravada pelo gerador
(`WT_peak`) tem que ser a energia $\widetilde W_T$ calculada por este notebook na mesma posição.

**Picos espúrios.** São marcadas como suspeitas as curvas com (i) algum pico com $\widetilde W_T>10^{30}$, valor que não corresponde a uma
ressonância física e indica perda de precisão no denominador; (ii) algum pico abaixo de $x_M=1{,}05\times10^{-4}$, colado na borda inferior da
janela; ou (iii) algum pico em $x_M$ muito menor que a primeira ressonância que a teoria permite. O item (iii) usa a ressonância de mais baixa
frequência, o modo monopolar do limite quase-estático (Eq. 4.45, Seção 8), escrita em função de $f$ com $O=1+\delta$:

$$
x_{M,1}^2=\frac{6\,O\,\alpha^f\beta^f}{\alpha^{-2(1-f)}\,\beta^f\big[O(\alpha^f+2\beta^f)-2\beta^f+2\big]+\beta^{-2(1-f)}\,\alpha^f\big[O^3(\beta^f+2)-3O\beta^f+2\beta^f-2\big]},
$$

e um pico é suspeito quando $x_M<0{,}1\,x_{M,1}$ (quando $x_{M,1}^2\le0$ o critério não se aplica). A lista vai para `tabelas/curvas_suspeitas.csv`.
**Por padrão nada é excluído** (`EXCLUIR_SUSPEITAS = False`); com `True`, as curvas suspeitas saem do banco antes de todos os ajustes.

In [ ]:
LIMITE_WT, FATOR_X1 = 1e30, 0.1
EXCLUIR_SUSPEITAS = False


def x1_monopolo(alpha, beta, delta, f):
    # Eq. (4.45) em f: rho_A/rho_M = alpha^f, rho_B/rho_M = beta^f, c_M/c_A = alpha^-(1-f), c_M/c_B = beta^-(1-f)
    a, b, d, f = np.broadcast_arrays(*(np.asarray(v, float) for v in (alpha, beta, delta, f)))
    rA, rB, mA, mB, O = a ** f, b ** f, a ** -(1 - f), b ** -(1 - f), 1 + d
    with np.errstate(all="ignore"):
        x2 = 6 * O * rA * rB / (mA ** 2 * rB * (O * (rA + 2 * rB) - 2 * rB + 2) + mB ** 2 * rA * (O ** 3 * (rB + 2) - 3 * O * rB + 2 * rB - 2))
    return np.where(x2 > 0, np.sqrt(np.abs(x2)), np.nan)


# energia: altura gravada contra a energia deste notebook, nos picos da malha base
ok = (MALHA["origem"] == "malha_base") & np.isfinite(MALHA["WT_peak"]) & (MALHA["WT_peak"] > 0) & np.isfinite(MALHA["Q_energy"]) & (MALHA["Q_energy"] < 1e3)
amostra = MALHA.loc[ok].sample(min(400, int(ok.sum())), random_state=42) if ok.any() else MALHA.iloc[:0]
dif = np.array([abs(energia_total([r.xM_peak], r.delta, materiais(r.alpha, r.beta, r.f))[0] / r.WT_peak - 1) for r in amostra.itertuples()])
if len(dif):
    print(f"energia em {len(dif)} picos bem resolvidos: diferença relativa mediana {np.median(dif):.1e}, máxima {dif.max():.1e}")
    if np.median(dif) > 1e-6:
        print("ATENÇÃO: a energia deste notebook não bate com a do gerador (normalização diferente de 'ZM'?)")

# picos espúrios
x1 = RESUMO.set_index("curva")
x1 = pd.Series(x1_monopolo(x1["alpha"], x1["beta"], x1["delta"], x1["f"]), index=x1.index)
motivos = []
for nome, df in (("contagem final", RES), ("malha", MALHA)):
    lim = FATOR_X1 * df["curva"].map(x1).to_numpy(float)
    for criterio, mk in ((f"W_T > {LIMITE_WT:.0e}", df["WT_peak"].to_numpy(float) > LIMITE_WT), ("x_M na borda inferior", df["xM_peak"].to_numpy(float) < 1.05 * X_MIN),
                         ("x_M < 0,1 x_M1 (Eq. 4.45)", np.isfinite(lim) & (df["xM_peak"].to_numpy(float) < lim))):
        if mk.any():
            motivos.append(df.loc[mk, ["curva", "alpha", "beta", "delta", "f", "xM_peak", "WT_peak"]].assign(arquivo=nome, criterio=criterio))
SUSPEITAS = pd.concat(motivos, ignore_index=True) if motivos else pd.DataFrame(columns=["curva", "criterio"])
SUSPEITAS.to_csv(os.path.join(TAB_DIR, "curvas_suspeitas.csv"), index=False)
CURVAS_SUSPEITAS = set(SUSPEITAS["curva"])
print(f"\ncurvas suspeitas: {len(CURVAS_SUSPEITAS)} de {len(RESUMO)}")
if len(SUSPEITAS):
    print(SUSPEITAS.groupby(["criterio", "arquivo"])["curva"].nunique().to_string())
    print(SUSPEITAS.drop_duplicates("curva").head(20).to_string(index=False, float_format=lambda v: f"{v:.4g}"))
if EXCLUIR_SUSPEITAS and CURVAS_SUSPEITAS:
    RESUMO, RES, MALHA = (d.loc[~d["curva"].isin(CURVAS_SUSPEITAS)].reset_index(drop=True) for d in (RESUMO, RES, MALHA))
    print(f"excluídas: {len(CURVAS_SUSPEITAS)} curvas; restam {len(RESUMO)}")
else:
    print("nenhuma curva excluída (EXCLUIR_SUSPEITAS = False)")

## 6. Mapas de estados ressonantes e espectros

Os mapas e os espectros desta seção ilustram três valores de $f$ das grades, $f=0$, $0{,}5$ e $1$, os dois extremos e o meio do intervalo. São apenas figuras: nenhum ajuste é feito por valor de $f$. Os mapas $Z_A\times Z_B$ com a fronteira da relação de existência
são desenhados na Seção 7, depois do ajuste.

### 6.1 Mapas $\delta\times\eta$

Número médio de ressonâncias (picos de $W_T$ na malha do gerador) por célula no plano $(\delta,\eta)$, com $\eta=(Z_A-Z_B)/(Z_A+Z_B)$. Os
quadrados marcam os espectros da Seção 6.2: $\eta\approx+0{,}7$ (vermelho), $\eta\approx0$ (cinza) e $\eta\approx-0{,}7$ (preto), com
$\delta=0{,}10$, $0{,}50$ e $1{,}35$.

In [ ]:
F_ILUSTRA = [0.0, 0.5, 1.0]
PARES_ESPECTRO = {"eta_positive": (r"$\eta_c\approx+0{,}7$", 0.006333333333333333, 0.001, BOX_COLOR),
                  "eta_zero": (r"$\eta_c\approx0$", 0.00811111111111111, 0.00811111111111111, "#B0B0B0"),
                  "eta_negative": (r"$\eta_c\approx-0{,}7$", 0.001, 0.006333333333333333, "black")}
DELTAS_ESPECTRO = [0.10, 0.50, 1.35]
DELTAS_GRADE = np.array([0.01, 0.05, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50, 0.60, 0.70, 0.80, 0.90, 1.00, 1.10, 1.20, 1.35, 1.50])
BINS_ETA = np.linspace(-1.0, 1.0, 21)


def grade_de(f):
    d = RESUMO.loc[(RESUMO["conjunto"] == "grade") & np.isclose(RESUMO["f"], f)]
    return d if len(d) else None


def par_mais_proximo(d, alpha, beta):
    pares = d[["alpha", "beta"]].drop_duplicates().to_numpy(float)
    i = int(np.argmin(np.sum((np.log(pares) - np.log([alpha, beta])) ** 2, axis=1)))
    return tuple(pares[i])


def mapa_delta_eta(d):
    idd = np.abs(d["delta"].to_numpy()[:, None] - DELTAS_GRADE[None, :]).argmin(axis=1)
    ie = np.clip(np.searchsorted(BINS_ETA, d["eta"].to_numpy(), side="left") - 1, 0, len(BINS_ETA) - 2)
    soma, n = np.zeros((len(DELTAS_GRADE), len(BINS_ETA) - 1)), np.zeros((len(DELTAS_GRADE), len(BINS_ETA) - 1))
    np.add.at(soma, (idd, ie), d["N_malha"].to_numpy(float))
    np.add.at(n, (idd, ie), 1)
    with np.errstate(invalid="ignore"):
        return np.where(n > 0, soma / n, np.nan)


with plt.rc_context(MAPA_RC):
    for f in F_ILUSTRA:
        d = grade_de(f)
        if d is None:
            print(f"f = {f:g}: sem grade")
            continue
        media = mapa_delta_eta(d)
        vmax = max(np.nanmax(media), 1.0)
        norm = Normalize(0.0, np.log10(1.0 + vmax))
        fig, ax, cax = figura_mapa(len(DELTAS_GRADE), len(BINS_ETA) - 1)
        desenhar_celulas(ax, media, norm)
        for nome, (_, a, b, cor) in PARES_ESPECTRO.items():
            a, b = par_mais_proximo(d, a, b)
            ie = int(np.clip(np.searchsorted(BINS_ETA, (a - b) / (a + b), side="left") - 1, 0, len(BINS_ETA) - 2))
            for dd in DELTAS_ESPECTRO:
                ax.add_patch(Rectangle((np.abs(DELTAS_GRADE - dd).argmin() - 0.5, ie - 0.5), 1, 1, fill=False, edgecolor=cor, linewidth=5, zorder=10))
        ax.axhline((0.0 - BINS_ETA[0]) / (BINS_ETA[1] - BINS_ETA[0]) - 0.5, color="#C7D0DB", linestyle="--", linewidth=1.35, zorder=5)
        tx = [int(np.abs(DELTAS_GRADE - v).argmin()) for v in (0.01, 0.2, 0.5, 0.8, 1.1, 1.5)]
        ax.set_xticks(tx); ax.set_xticklabels([virgula(f"{DELTAS_GRADE[i]:g}") for i in tx])
        te = np.array([-1.0, -0.5, 0.0, 0.5, 1.0])
        ax.set_yticks((te - BINS_ETA[0]) / (BINS_ETA[1] - BINS_ETA[0]) - 0.5); ax.set_yticklabels([virgula(f"{v:.1f}") for v in te])
        ax.set_xlabel(r"$\delta$", fontsize=23, labelpad=9)
        ax.set_ylabel(r"$\eta=(Z_A-Z_B)/(Z_A+Z_B)$", fontsize=23, labelpad=10)
        ax.set_xlim(-0.5, len(DELTAS_GRADE) - 0.5); ax.set_ylim(-0.5, len(BINS_ETA) - 1.5)
        ax.set_aspect("equal", adjustable="box")
        caixa_mapa(ax, r"$\left\langle N_{\mathrm{res}}\right\rangle$")
        ticks = np.unique(np.r_[0, [v for v in (1, 2, 5, 10, 20, 50, 100, 200, 500) if v < 0.8 * vmax], vmax])
        cbar = barra_mapa(fig, cax, norm, np.log10(1 + ticks), [virgula(f"{t:.3g}") for t in ticks], "Ressonâncias")
        print(f"f = {f:g}: N médio máximo por célula = {np.nanmax(media):.3g}")
        concluir_mapa(fig, ax, cbar, f"mapa_delta_eta_f{f:.2f}")

### 6.2 Espectros e tabelas

$W_T$ na malha reconstruída do gerador para $\eta\approx+0{,}7$, $0$ e $-0{,}7$ e $\delta=0{,}10$, $0{,}50$ e $1{,}35$ (os quadrados da Seção 6.1),
com os picos gravados pelo gerador marcados. A malha é refeita a partir do que ficou no checkpoint (polos, auditoria e modos por $\ell$), com as
mesmas regras do gerador: malha base e trechos uniformes em volta de cada grupo de picos, com o passo necessário para ver cada um. A tabela de cada
caso traz, a partir dos picos gravados até $x_M=1$, a primeira ressonância (posição, FWHM e $Q$), a mediana do espaçamento entre picos
consecutivos (macro-FSR) e as médias de FWHM e de $Q$.

In [ ]:
# regras do gerador (versão 8) usadas para refazer a malha de uma curva a partir do checkpoint
N_LOG, N_MEIO, N_FIM, PROM_WT, PROM_WL, Q_MIN, DX_MIN, N_MAX_PONTOS = 1000, 3000, 1000, 0.1, 0.75, 2.0, 1e-9, 2_000_000


def malha_base(delta, mat, x_min=X_MIN, x_max=X_MAX):
    rM, rA, rB, cM, cA, cB = mat
    esp = np.pi / (cM / cA + delta * cM / cB)
    ext = min(esp, 0.05 * x_max)
    x = np.unique(np.concatenate([np.geomspace(x_min, 0.1, N_LOG), np.linspace(0.1, 0.9, N_MEIO), np.linspace(0.9, x_max + ext, N_FIM)]))
    return x, esp


def maximos(energia, x, esp, prom=None, w=None):
    w = energia(x) if w is None else w
    if prom is None:
        idx = find_peaks(np.where(np.isfinite(w), w, -np.inf))[0]
    else:
        ok = np.isfinite(w) & (w > 0)
        if not ok.any():
            return np.array([], int), np.array([])
        piso = max(w[ok].min() * 1e-14, np.finfo(float).tiny)
        idx = find_peaks(np.log10(np.where(ok, np.maximum(w, piso), piso)), prominence=prom)[0]
    fica, xs = [], []
    for i in idx:
        xp = float(x[i])
        if xp > X_MAX - 2 * min(esp, 0.05 * X_MAX) and 0 < i < len(x) - 1:
            alvo = lambda t: -np.log(max(float(energia(np.array([t]))[0]), np.finfo(float).tiny))
            xp = float(minimize_scalar(alvo, bounds=(x[i - 1], x[i + 1]), method="bounded", options={"xatol": 1e-12}).x)
        if X_MIN <= xp <= X_MAX:
            fica.append(i); xs.append(xp)
    return np.asarray(fica, int), np.asarray(xs, float)


def refina_posicoes(energia, x, xs, it=60):
    if len(xs) == 0:
        return xs
    i = np.clip(np.searchsorted(x, xs), 1, len(x) - 1)
    a, b, g = x[i - 1], x[np.minimum(i + 1, len(x) - 1)], (np.sqrt(5) - 1) / 2
    c, d = b - g * (b - a), a + g * (b - a)
    fc, fd = energia(c), energia(d)
    for _ in range(it):
        m = np.nan_to_num(fc, nan=-np.inf) > np.nan_to_num(fd, nan=-np.inf)
        b, a = np.where(m, d, b), np.where(m, a, c)
        c, d = b - g * (b - a), a + g * (b - a)
        fc, fd = energia(c), energia(d)
    return 0.5 * (a + b)


def casar(xs, zeros, esp):
    pares = sorted((abs(x - z.real), i, j) for i, x in enumerate(xs) for j, z in enumerate(zeros) if abs(x - z.real) <= max(2 * abs(z.imag), 0.25 * esp))
    usado_x, usado_z, casados = set(), set(), {}
    for _, i, j in pares:
        if i not in usado_x and j not in usado_z:
            usado_x.add(i); usado_z.add(j); casados[j] = i
    return casados


def modos_por_l(delta, mat, zeros_por_l, x, esp):
    saida = []
    for l in range(LMAX + 1):
        zeros, E = zeros_por_l.get(l), (lambda v, l=l: energia_modal(l, v, delta, mat)[0])
        _, xs = maximos(E, x, esp)
        r = {"l": l, "N_zeros": None if zeros is None else len(zeros)}
        if zeros is not None:
            cas = casar(xs, zeros, esp)
            if len(xs) == len(zeros) == len(cas):
                saida.append(dict(r, regra="bate", x=[z.real for z in zeros])); continue
            bons = [j for j, z in enumerate(zeros) if z.real / (2 * abs(z.imag)) >= Q_MIN]
            sobra_x = [v for i, v in enumerate(xs) if i not in {cas[j] for j in cas if j not in bons}]
            z2 = [zeros[j] for j in bons]
            if len(sobra_x) == len(z2) == len(casar(sobra_x, z2, esp)):
                saida.append(dict(r, regra="bate_Q2", x=[z.real for z in z2])); continue
        _, xp = maximos(E, x, esp, prom=PROM_WL)
        saida.append(dict(r, regra="proeminencia" if zeros is not None else "sem_polos", x=list(refina_posicoes(E, x, xp))))
    return saida


def separacao_laurent(polos, delta, mat, n=400):
    # perto de dois polos vizinhos, 1/D_l ~ A1/(x - z1) + A2/(x - z2); distância entre os máximos desse modelo (nan se fundem)
    pol = sorted(polos, key=lambda p: p["pole_real"])
    z, l = np.array([complex(p["pole_real"], p["pole_imag"]) for p in pol]), np.array([p["l"] for p in pol])
    A, h = np.empty(len(z), complex), np.empty(len(z))
    for ll in np.unique(l):
        k = l == ll
        zz, e = z[k], 1e-7 * np.maximum(np.abs(z[k]), 1e-4)
        A[k] = 2 * e / ((zz + e) ** 4 * coeficientes(ll, zz + e, delta, mat)[4] - (zz - e) ** 4 * coeficientes(ll, zz - e, delta, mat)[4])
        gam = np.abs(zz.imag); s_ = np.maximum(gam, 1e-6 * zz.real)
        h[k] = energia_modal(ll, zz.real + s_, delta, mat)[0] * (s_ ** 2 + gam ** 2) / gam ** 2
    z1, z2 = z[:-1], z[1:]
    x1, x2, G1, G2 = z1.real, z2.real, np.abs(z1.imag)[:, None], np.abs(z2.imag)[:, None]
    g1, g2 = np.maximum(G1[:, 0], 1e3 * np.finfo(float).eps * x1), np.maximum(G2[:, 0], 1e3 * np.finfo(float).eps * x2)
    u = np.geomspace(1e-3, 30, n // 4)
    X = np.sort(np.concatenate([x1[:, None] - g1[:, None] * u[::-1], x1[:, None] + g1[:, None] * u, np.linspace(x1, x2, n // 4 + 2, axis=1)[:, 1:-1],
                                x2[:, None] - g2[:, None] * u[::-1], x2[:, None] + g2[:, None] * u], axis=1), axis=1)
    with np.errstate(all="ignore"):
        W = np.where((l[:-1] == l[1:])[:, None], np.abs(A[:-1, None] / (X - z1[:, None]) + A[1:, None] / (X - z2[:, None])) ** 2,
                     h[:-1, None] * G1 ** 2 / ((X - x1[:, None]) ** 2 + G1 ** 2) + h[1:, None] * G2 ** 2 / ((X - x2[:, None]) ** 2 + G2 ** 2))
    mx = (W[:, 1:-1] > W[:, :-2]) & (W[:, 1:-1] >= W[:, 2:]) & np.isfinite(W[:, 1:-1])
    sep = np.full(len(z1), np.nan)
    for i in np.flatnonzero(mx.sum(axis=1) >= 2):
        p = X[i, 1:-1][mx[i]]; sep[i] = p.max() - p.min()
    return 0.5 * (x1 + x2), sep


def malha_dx(x, alvos, esp, laurent=None):
    A = sorted([a for a in alvos if np.isfinite(a[1]) and a[1] > 0], key=lambda a: a[0])
    if not A:
        return x
    P, Dn, ext, seg = np.array([a[0] for a in A]), np.array([a[1] for a in A]), min(esp, 0.05 * X_MAX), []
    for gi in np.split(np.arange(len(P)), np.flatnonzero(np.diff(P) > 0.5 * esp) + 1):
        lo, hi, d = max(P[gi[0]] - 0.25 * esp, X_MIN), min(P[gi[-1]] + 0.25 * esp, X_MAX + ext), float(Dn[gi].min())
        if laurent is not None and len(laurent[0]):
            k = (laurent[0] >= lo) & (laurent[0] <= hi) & np.isfinite(laurent[1]) & (laurent[1] > 0)
            if k.any():
                d = min(d, 0.5 * float(laurent[1][k].min()))
        seg.append((lo, hi, d))
    livre, custo, piso = max(N_MAX_PONTOS - len(x), 1000), (lambda p: sum((hi - lo) / max(d, p) for lo, hi, d in seg)), DX_MIN
    if custo(piso) > livre:
        a, b = DX_MIN, max(hi - lo for lo, hi, _ in seg)
        for _ in range(60):
            m = np.sqrt(a * b); a, b = (m, b) if custo(m) > livre else (a, m)
        piso = b
    return np.unique(np.concatenate([x] + [lo + max(d, piso) * np.arange(int((hi - lo) / max(d, piso)) + 1) for lo, hi, d in seg]))


def reconstruir_malha(item, f):
    # malha usada pelo gerador nesta curva: base nas curvas normais, refinada pelos polos nas curvas de risco
    alpha, beta, delta = float(item["alpha"]), float(item["beta"]), float(item["delta"])
    mat = materiais(alpha, beta, f)
    x, esp = malha_base(delta, mat)
    if not bool(item.get("risco", alpha < 1 or beta < 1)):
        return x, esp, mat
    cand, aud = [dict(p) for p in item.get("candidatos", []) or []], item.get("auditoria", {}) or {}
    zeros_por_l = {l: [complex(p["pole_real"], p["pole_imag"]) for p in cand if int(p.get("l", -1)) == l and p.get("argument_audit_ok", False)]
                   if aud.get(l, aud.get(str(l))) == "ok" else None for l in range(LMAX + 1)}
    modos = modos_por_l(delta, mat, zeros_por_l, x, esp)
    bons = [p for p in cand if p.get("argument_audit_ok", False)]
    laurent = separacao_laurent(bons, delta, mat) if len(bons) >= 2 else None
    alvos = [(float(p["pole_real"]), float(p["dx_nec"]), None) for p in bons if p.get("Q_pole", 0) >= Q_MIN and p.get("prom_log10_pole", 0) >= PROM_WT
             and np.isfinite(p.get("dx_nec", np.nan)) and p.get("dx_nec", 0) > 0]
    for m in modos:
        if m["N_zeros"] is None or m["regra"] in ("proeminencia", "sem_polos"):
            P = np.sort(np.asarray(m["x"], float))
            if len(P):
                viz = np.minimum(np.r_[np.inf, np.diff(P)], np.r_[np.diff(P), np.inf])
                alvos += [(float(v), 0.5 * min(float(dv), 0.25 * esp), None) for v, dv in zip(P, viz)]
    return malha_dx(x, alvos, esp, laurent), esp, mat


def ler_checkpoint(f):
    a = GRADES.get(f)
    if a is None or a["ckpt"] is None:
        return None
    with np.load(a["ckpt"], allow_pickle=True) as d:
        curvas = d["curves"].item()
    return pd.DataFrame([dict(alpha=float(c["alpha"]), beta=float(c["beta"]), delta=float(c["delta"]), item=c) for c in curvas.values() if c.get("status") == "ok"])


def resumo_picos(item):
    # primeira ressonância, macro-FSR e médias de FWHM e Q dos picos gravados até x_M = 1
    p = sorted([r for r in item.get("ressonancias_malha", []) or [] if np.isfinite(r.get("xM_peak", np.nan)) and X_MIN <= r["xM_peak"] <= X_MAX], key=lambda r: r["xM_peak"])
    if not p:
        return None
    x, fw, q = (np.array([r.get(k, np.nan) for r in p], float) for k in ("xM_peak", "FWHM_energy", "Q_energy"))
    media_dp = lambda v: (np.mean(v[np.isfinite(v)]), np.std(v[np.isfinite(v)], ddof=1) if np.isfinite(v).sum() > 1 else 0.0) if np.isfinite(v).any() else (np.nan, np.nan)
    return dict(x1=x[0], fwhm1=fw[0], q1=q[0], fsr=np.median(np.diff(x)) if len(x) > 1 else np.nan, fwhm=media_dp(fw), q=media_dp(q))


def sci(v, sig=3):
    if not np.isfinite(v):
        return "--"
    e = int(np.floor(np.log10(abs(v)))) if v != 0 else 0
    return f"${dec(v / 10 ** e, sig - 1)}\\times10^{{{e}}}$"


def num(v, sig=3):
    return "--" if not np.isfinite(v) else "$" + gnum(v, sig) + "$"


def tabela_espectro(nome, resumos):
    linhas = ["1º pico\n($x_M$)", "1º pico\nFWHM", "1º pico\nFator $Q$", "Macro-FSR\n($\\Delta x_M$)",
              "FWHM médio\n($\\mu\\pm\\sigma$) [$\\times10^{-4}$]", "Fator $Q$ médio\n($\\mu\\pm\\sigma$)"]
    col = lambda s: ["--"] * 6 if s is None else [num(s["x1"], 5), sci(s["fwhm1"]), num(s["q1"]), sci(s["fsr"]),
                                                  f"${dec(1e4 * s['fwhm'][0], 2)}$\n$\\pm{dec(1e4 * s['fwhm'][1], 2)}$", sci(s["q"][0], 2) + "\n$\\pm$" + sci(s["q"][1], 2)]
    cols = [col(s) for s in resumos]
    salvar_tabela(nome, linhas, [f"$\\delta={dec(d, 2)}$" for d in DELTAS_ESPECTRO], [[c[i] for c in cols] for i in range(6)], figsize=(6.4, 4.5),
                  first_col_width=0.36, fontsize=15, header_label="Parâmetro",
                  csv=pd.DataFrame([dict(delta=d, **({} if s is None else {k: v for k, v in s.items() if not isinstance(v, tuple)})) for d, s in zip(DELTAS_ESPECTRO, resumos)]))


ZOOM = {0.0: [0.0, 0.0025, 0.005, 0.0075, 0.01], 0.5: [0.0, 0.005, 0.01, 0.015, 0.02], 1.0: [0.0, 0.2, 0.4, 0.6, 0.8, 1.0]}
for f in F_ILUSTRA:
    cps = ler_checkpoint(f)
    if cps is None or not len(cps):
        print(f"f = {f:g}: sem checkpoint")
        continue
    x_zoom = ZOOM[f][-1]
    for nome, (rot, a, b, _) in PARES_ESPECTRO.items():
        a, b = par_mais_proximo(cps, a, b)
        par = cps.loc[np.isclose(cps["alpha"], a) & np.isclose(cps["beta"], b)]
        linhas = [par.iloc[int(np.argmin(np.abs(par["delta"].to_numpy() - d)))] for d in DELTAS_ESPECTRO]
        print(f"f = {f:g} | {nome}: alpha = {a:.4g}, beta = {b:.4g}, eta = {(a - b) / (a + b):+.3f} | delta = " + ", ".join(f"{r['delta']:g}" for r in linhas))
        fig, ax = plt.subplots(figsize=(5, 3.5))
        for i, r in enumerate(linhas):
            xr, esp, mat = reconstruir_malha(r["item"], f)
            k = (xr >= X_MIN) & (xr <= x_zoom)
            wr = energia_total(xr[k], r["delta"], mat)
            ax.plot(xr[k], wr, color=COLORS[i], linestyle=LINESTYLES[i], linewidth=LINEWIDTHS[i], label=rf"$\delta={dec(r['delta'], 2)}$")
            xs = [p["xM_peak"] for p in r["item"].get("ressonancias_malha", []) or [] if X_MIN <= p.get("xM_peak", np.nan) <= x_zoom]
            j = [int(np.clip(np.searchsorted(xr[k], v), 1, k.sum() - 2)) for v in xs]
            j = [jj - 4 + int(np.argmax(wr[max(jj - 4, 0):jj + 5])) if jj >= 4 else jj for jj in j]    # máximo local da malha mais próximo
            ax.scatter(xr[k][j], wr[j], s=16, color=COLORS[i], edgecolors="white", linewidths=0.45, zorder=5)
            print(f"   delta = {r['delta']:.2f}: {len(xr)} pontos de malha, {len(xs)} picos gravados até x_M = {x_zoom:g}")
        eixo_x_fixo(ax, ZOOM[f])
        eixo_log_y(ax, [1e-7, 1e-3, 1e1, 1e5, 1e9], (1e-7, 1e10))
        estilizar(ax, r"$x_M$")
        adicionar_legenda(ax)
        adicionar_caixa(ax, rot)
        fig.subplots_adjust(left=0.16, right=0.985, bottom=0.15, top=0.98)
        concluir(fig, f"espectro_f{f:.2f}_{nome}")
        tabela_espectro(f"tabela_espectro_f{f:.2f}_{nome}", [resumo_picos(r["item"]) for r in linhas])

## 7. Relação de existência

Uma configuração é ressonante quando tem ao menos uma ressonância em $10^{-4}\le x_M\le1$. Com o contraste repartido na forma
$\rho_A/\rho_M=\alpha^f$, $c_A/c_M=\alpha^{1-f}$, $\rho_B/\rho_M=\beta^f$ e $c_B/c_M=\beta^{1-f}$, uma onda de frequência adimensional $x_M$ acumula a fase
$x_M\,c_M/c_A$ ao atravessar o núcleo e $x_M\,\delta\,c_M/c_B$ ao atravessar a casca, de modo que a fase por unidade de $x_M$ é (Eq. 4.18)

$$
\Phi=\frac{c_M}{c_A}+\delta\,\frac{c_M}{c_B}=\alpha^{-(1-f)}+\delta\,\beta^{-(1-f)}=e^{u},\qquad
u=\ln\Big[\alpha^{-(1-f)}+\delta\,\beta^{-(1-f)}\Big].
$$

Como o número de ressonâncias abaixo de $x_M$ é aproximadamente $\Phi x_M/\pi$, a primeira aparece em $x_M\le1$ quando (Eq. 4.42)

$$
\alpha^{-(1-f)}+\delta\,\beta^{-(1-f)}\ge\pi\qquad(u\ge\ln\pi),
$$

critério sem nenhum coeficiente, que vale para $f\to0$. Para $f>0$ a razão de densidades também decide se há
ressonância, pelo mecanismo do tipo Minnaert. Por isso a relação de existência é **uma única** regressão logística para todo $0\le f\le1$, com $f$
explícito nos termos e a variável $u(\alpha,\beta,\delta,f)$ como ponto de partida (Eqs. 4.17 e 4.19):

$$
P_{\mathrm{res}}(\alpha,\beta,\delta,f)=\frac{1}{1+e^{-g}},\qquad
g=c_0+c_1u+c_2u^2+c_3\ln\alpha+c_4\delta+c_5(\ln\beta)^2+c_6\ln\alpha\ln\beta+c_7f\ln\alpha+c_8\delta\ln\beta+c_9\delta f,
$$

e a configuração é ressonante quando $P_{\mathrm{res}}\ge1/2$ ($g\ge0$). **Os coeficientes $c_i$ são ajustados aqui** a todo o banco (grades e
configurações sorteadas, cada uma com o seu $f$), por máxima verossimilhança com a regularização L2 padrão do scikit-learn sobre as variáveis
padronizadas; as incertezas vêm da informação de Fisher. **Teste:** $20\,\%$ das curvas, sorteadas uma única vez com a semente 42, ficam fora do
ajuste; são as mesmas curvas de teste da relação de recorrência (Seção 8). As métricas podem ser vistas em função de $f$, mas o ajuste é um só.

In [ ]:
TEST_FRACTION, RANDOM_SEED = 0.20, 42
TEX_TERMOS = [r"$1$", r"$u$", r"$u^2$", r"$\ln\alpha$", r"$\delta$", r"$(\ln\beta)^2$", r"$\ln\alpha\,\ln\beta$", r"$f\,\ln\alpha$", r"$\delta\,\ln\beta$", r"$\delta f$"]


def variavel_u(alpha, beta, delta, f):
    # u = ln[alpha^-(1-f) + delta beta^-(1-f)]   (Eq. 4.18)
    alpha, beta, delta, f = (np.asarray(v, float) for v in (alpha, beta, delta, f))
    return np.log(alpha ** -(1 - f) + delta * beta ** -(1 - f))


def termos_existencia(alpha, beta, delta, f):
    # [1, u, u^2, ln a, delta, (ln b)^2, ln a ln b, f ln a, delta ln b, delta f]   (Eq. 4.19)
    alpha, beta, delta, f = np.broadcast_arrays(*(np.atleast_1d(np.asarray(v, float)) for v in (alpha, beta, delta, f)))
    u, la, lb = variavel_u(alpha, beta, delta, f), np.log(alpha), np.log(beta)
    return np.column_stack([np.ones_like(u), u, u * u, la, delta, lb * lb, la * lb, f * la, delta * lb, delta * f])


def p_res(alpha, beta, delta, f, coef):
    return 1.0 / (1.0 + np.exp(-np.clip(termos_existencia(alpha, beta, delta, f) @ coef, -700, 700)))


def ajusta_logistica(X, y):
    # X sem a coluna de uns; ajuste com as variáveis padronizadas (L2 padrão do scikit-learn) e volta para as variáveis originais
    mu, sd = X.mean(axis=0), X.std(axis=0)
    sd[sd == 0] = 1.0
    lr = LogisticRegression(max_iter=20000).fit((X - mu) / sd, y)
    coef = np.r_[lr.intercept_[0] - np.sum(lr.coef_[0] * mu / sd), lr.coef_[0] / sd]
    Xa = np.column_stack([np.ones(len(X)), X])
    p = 1 / (1 + np.exp(-np.clip(Xa @ coef, -700, 700)))
    return coef, np.sqrt(np.clip(np.diag(np.linalg.pinv(Xa.T @ (Xa * (p * (1 - p))[:, None]))), 0, None))


def metricas_existencia(df, coef=None):
    y = df["ressonante"].to_numpy()
    p = p_res(df["alpha"], df["beta"], df["delta"], df["f"], COEF_EXIST if coef is None else coef)
    pred = (p >= 0.5).astype(int)
    return {"N": len(y), "fracao": float(y.mean()) if len(y) else np.nan, "acuracia": accuracy_score(y, pred), "precisao": precision_score(y, pred, zero_division=0),
            "revocacao": recall_score(y, pred, zero_division=0), "F1": f1_score(y, pred, zero_division=0),
            "AUC": roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan, "FP": int(((pred == 1) & (y == 0)).sum()), "FN": int(((pred == 0) & (y == 1)).sum())}


def faixas_f(df, n=5):
    # faixas de f só para mostrar as métricas em função de f (o ajuste é um só)
    bordas = np.linspace(0, 1, n + 1)
    i = np.clip(np.searchsorted(bordas, df["f"].to_numpy(float), side="right") - 1, 0, n - 1)
    return i, 0.5 * (bordas[:-1] + bordas[1:])


indices = np.random.default_rng(RANDOM_SEED).permutation(len(RESUMO))
n_teste = int(round(TEST_FRACTION * len(RESUMO)))
treino_c, teste_c = RESUMO.iloc[indices[n_teste:]].reset_index(drop=True), RESUMO.iloc[indices[:n_teste]].reset_index(drop=True)
CURVAS_TESTE = set(teste_c["curva"])

COEF_EXIST, ERRO_EXIST = ajusta_logistica(termos_existencia(treino_c["alpha"], treino_c["beta"], treino_c["delta"], treino_c["f"])[:, 1:],
                                          treino_c["ressonante"].to_numpy())
MET_EXIST = {"treino": metricas_existencia(treino_c), "teste": metricas_existencia(teste_c)}
print(f"treino: {len(treino_c)} configurações | teste (20 % das curvas, semente 42): {len(teste_c)}\n")
for i, t in enumerate(["1", "u", "u^2", "ln a", "delta", "(ln b)^2", "ln a ln b", "f ln a", "delta ln b", "delta f"]):
    aviso = "" if abs(COEF_EXIST[i]) >= 2 * ERRO_EXIST[i] else "   <- |c| < 2 sigma"
    print(f"c{i} ({t:>10s}) = {COEF_EXIST[i]:+.5e} ± {ERRO_EXIST[i]:.2e}{aviso}")
for parte, m_ in MET_EXIST.items():
    print(parte, {k: round(v, 4) if isinstance(v, float) else v for k, v in m_.items()})
i_f, F_CENTRO = faixas_f(teste_c)
MET_F = pd.DataFrame([dict(f=F_CENTRO[k], **metricas_existencia(teste_c.loc[i_f == k])) for k in range(len(F_CENTRO)) if (i_f == k).sum() >= 2])
MET_F["criterio_4_42"] = [np.mean((teste_c.loc[i_f == k, "Phi"] >= np.pi) == teste_c.loc[i_f == k, "ressonante"]) for k in range(len(F_CENTRO)) if (i_f == k).sum() >= 2]
print("\nteste em função de f (faixas de largura 0,2):")
print(MET_F[["f", "N", "acuracia", "revocacao", "AUC", "criterio_4_42"]].to_string(index=False, float_format=lambda v: f"{v:.4f}"))

### Desempenho no teste

Probabilidade prevista para cada configuração de teste contra a existência observada (com um pequeno espalhamento horizontal para separar os
pontos), e as mesmas métricas em função de $f$, junto com o critério sem coeficientes $\Phi\ge\pi$ (Eq. 4.42).

In [ ]:
y = teste_c["ressonante"].to_numpy(float)
p = p_res(teste_c["alpha"], teste_c["beta"], teste_c["delta"], teste_c["f"], COEF_EXIST)
xj = y + np.random.default_rng(RANDOM_SEED).uniform(-0.05, 0.05, len(y))
m_ = MET_EXIST["teste"]

fig, ax = plt.subplots(figsize=(5, 3.5))
ax.scatter(xj[y == 0], p[y == 0], s=14, facecolors="white", edgecolors="#6488A5", linewidths=0.7, zorder=2, label="Sem ressonância")
ax.scatter(xj[y == 1], p[y == 1], s=14, facecolors="#8B1A1A", edgecolors="black", linewidths=0.4, zorder=3, label="Com ressonância")
ax.axhline(0.5, color="black", linestyle=":", linewidth=1.2, zorder=1, label=rf"$P_{{\mathrm{{res}}}}=0{VIRG}5$")
ax.set_xlim(-0.15, 1.15)
ax.set_ylim(-0.03, 1.03)
ax.set_xticks([0, 1])
ax.set_xticklabels([r"$0$", r"$1$"])
ax.set_yticks([0, 0.25, 0.5, 0.75, 1.0])
ax.set_yticklabels([r"$0$", rf"$0{VIRG}25$", rf"$0{VIRG}50$", rf"$0{VIRG}75$", r"$1$"])
estilizar(ax, "Existência observada", r"$P_{\mathrm{res}}$ previsto")
adicionar_caixa(ax, f"$N={milhar(m_['N'])}$\n" + f"Acurácia $={dec(m_['acuracia'], 3)}$\n" + f"Precisão $={dec(m_['precisao'], 3)}$\n"
                + f"Revocação $={dec(m_['revocacao'], 3)}$\n" + f"ROC AUC $={dec(m_['AUC'], 4)}$", x=0.035, ha="left")
adicionar_legenda(ax)
fig.subplots_adjust(left=0.14, right=0.985, bottom=0.15, top=0.98)
concluir(fig, "grafico_existencia")

fig, ax = plt.subplots(figsize=(5, 3.5))
for i, (col, rot) in enumerate((("acuracia", "Acurácia"), ("revocacao", "Revocação"), ("AUC", "ROC AUC"), ("criterio_4_42", r"$\Phi\geq\pi$ (Eq. 4.42)"))):
    ax.plot(MET_F["f"], MET_F[col], color=COLORS[i], linestyle=LINESTYLES[i], linewidth=LINEWIDTHS[i], marker="o", markersize=3, label=rot)
eixo_x_fixo(ax, [0.0, 0.25, 0.5, 0.75, 1.0])
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"${dec(v, 2)}$"))
estilizar(ax, r"$f$", "Métrica no teste")
adicionar_legenda(ax, loc="lower left")
fig.subplots_adjust(left=0.14, right=0.985, bottom=0.15, top=0.98)
concluir(fig, "grafico_existencia_contra_f")

### Probabilidade de ressonância, $P_{\mathrm{res}}$

$P_{\mathrm{res}}$ da relação de existência no plano $(\alpha,\beta)$ com $\delta=0{,}5$, para $f=1$, $0{,}5$ e $0$ (a mesma relação, avaliada em três
valores de $f$). A linha vermelha é a fronteira logística $P_{\mathrm{res}}=0{,}5$ e a azul, a fronteira observada na grade do mesmo $f$.

In [ ]:
def mapa_binado(df, valor="N"):
    # maior valor entre as curvas de cada célula; células vazias recebem o valor do vizinho mais próximo
    e = np.linspace(*LOG_Z, NZ + 1)
    la, lb, n = np.log10(df["alpha"].to_numpy()), np.log10(df["beta"].to_numpy()), df[valor].to_numpy(float)
    ia = np.clip(np.searchsorted(e, la, side="right") - 1, 0, NZ - 1)
    ib = np.clip(np.searchsorted(e, lb, side="right") - 1, 0, NZ - 1)
    contagem, maximo = np.zeros((NZ, NZ), int), np.full((NZ, NZ), -np.inf)
    np.add.at(contagem, (ia, ib), 1)
    np.maximum.at(maximo, (ia, ib), n)
    vazio = contagem == 0
    if vazio.any():
        ga, gb = np.meshgrid(0.5 * (e[:-1] + e[1:]), 0.5 * (e[:-1] + e[1:]), indexing="ij")
        _, viz = cKDTree(np.column_stack([la, lb])).query(np.column_stack([ga[vazio], gb[vazio]]))
        maximo[vazio] = n[viz]
    return np.maximum(maximo, 0.0)


def na_grade(d, delta):
    dd = float(np.unique(d["delta"])[np.argmin(np.abs(np.unique(d["delta"]) - delta))])
    return d.loc[np.isclose(d["delta"], dd)], dd


def eixos_alpha_beta(ax, xlabel=r"$\alpha=Z_A/Z_M$", ylabel=r"$\beta=Z_B/Z_M$"):
    t = 10.0 ** np.arange(LOG_Z[0], LOG_Z[1] + 1e-9)
    fmt = lambda v: f"{v:.4f}".rstrip("0").rstrip(".").replace(".", ",") if v < 1 else f"{v:g}"
    ax.set_xticks(celula(t)); ax.set_xticklabels([fmt(v) for v in t])
    ax.set_yticks(celula(t)); ax.set_yticklabels([fmt(v) for v in t])
    ax.set_xlabel(xlabel, fontsize=23, labelpad=9)
    ax.set_ylabel(ylabel, fontsize=23, labelpad=10)
    ax.set_xlim(-0.5, NZ - 0.5); ax.set_ylim(-0.5, NZ - 0.5)
    ax.set_aspect("equal", adjustable="box")


def fronteira_logistica(ax, delta, f):
    GA, GB = np.meshgrid(np.linspace(*LOG_Z, 400), np.linspace(*LOG_Z, 400), indexing="ij")
    P = p_res(10 ** GA.ravel(), 10 ** GB.ravel(), delta, f, COEF_EXIST).reshape(GA.shape)
    ax.contour(celula(10 ** GA), celula(10 ** GB), P, levels=[0.5], colors=[FRONT_LOG["color"]], linewidths=FRONT_LOG["linewidth"],
               linestyles=[FRONT_LOG["linestyle"]], alpha=FRONT_LOG["alpha"], zorder=18)
    return GA, GB, P


DELTA_PRES = 0.5
with plt.rc_context(MAPA_RC):
    for f in F_ILUSTRA[::-1]:
        fig, ax, cax = figura_mapa()
        GA, GB, P = fronteira_logistica(ax, DELTA_PRES, f)
        ax.pcolormesh(celula(10 ** GA), celula(10 ** GB), P, cmap=RESONANCE_CMAP, vmin=0, vmax=1, shading="auto", rasterized=True, zorder=1)
        handles = [Line2D([0], [0], label="Fronteira logística\n" + r"($P_{\mathrm{res}}=0{,}5$)", **FRONT_LOG)]
        d = grade_de(f)
        if d is not None:
            sub, dd = na_grade(d, DELTA_PRES)
            desenhar_fronteira(ax, mapa_binado(sub) > 0, FRONT_OBS)
            handles.insert(0, Line2D([0], [0], label="Fronteira observada", **FRONT_OBS))
            acerto = np.mean((p_res(sub["alpha"], sub["beta"], sub["delta"], f, COEF_EXIST) >= 0.5) == sub["ressonante"])
            print(f"f = {f:g}, delta = {dd:g}: P_res >= 0,5 concorda com a grade em {100 * acerto:.1f} % das {len(sub)} curvas")
        eixos_alpha_beta(ax)
        legenda_mapa(ax, handles, loc="lower left", bbox=(0.015, 0.015))
        caixa_mapa(ax, rf"$\delta={dec(DELTA_PRES, 1)},\ f={virgula(f'{f:g}')}$", x=0.045, ha="left")
        caixa_mapa(ax, r"$P_{\mathrm{res}}$")
        cbar = barra_mapa(fig, cax, Normalize(0, 1), [0, 0.25, 0.5, 0.75, 1], ["0", "0,25", "0,5", "0,75", "1"], "Probabilidade de ressonância")
        concluir_mapa(fig, ax, cbar, f"mapa_Pres_delta{DELTA_PRES:g}_f{f:.2f}")

### Mapas de estados ressonantes $Z_A\times Z_B$

Número de ressonâncias (picos de $W_T$ na malha do gerador) até $x_M=1$ no plano $(\alpha,\beta)$ para $\delta=1{,}5$, nos três valores de $f$ da
Seção 6, com a estética do mapa de estados da esfera: cada célula mostra o maior número de ressonâncias entre as curvas que ela contém, e células
vazias recebem o valor do vizinho mais próximo. Estão desenhadas a fronteira observada para $\delta=1{,}5$ (azul), as fronteiras observadas
de referência para $\delta=0{,}01$ e $0{,}5$ e a fronteira logística $P_{\mathrm{res}}=0{,}5$ da relação de existência para $\delta=1{,}5$ (vermelha).

In [ ]:
DELTA_MAPA, DELTAS_REF = 1.50, [(0.01, dict(color="0.45", linestyle=(0, (4.5, 2.5)), linewidth=1.9, alpha=0.92)),
                                (0.50, dict(color="black", linestyle=(0, (5.0, 2.2)), linewidth=2.0, alpha=0.82))]
ROTULOS_EIXOS = {0.0: (r"$\alpha_v=c_A/c_M$", r"$\beta_v=c_B/c_M$"), 0.5: (r"$\alpha=Z_A/Z_M$", r"$\beta=Z_B/Z_M$"), 1.0: (r"$\alpha_d=\rho_A/\rho_M$", r"$\beta_d=\rho_B/\rho_M$")}
with plt.rc_context(MAPA_RC):
    for f in F_ILUSTRA:
        d = grade_de(f)
        if d is None:
            continue
        sub, dd = na_grade(d, DELTA_MAPA)
        mapa = mapa_binado(sub, "N_malha")
        n_max = max(int(np.ceil(mapa.max())), 1)
        norm = Normalize(vmin=0.0, vmax=np.log10(1.0 + n_max))
        fig, ax, cax = figura_mapa()
        desenhar_celulas(ax, mapa, norm)
        um = celula(1.0)
        ax.plot([-0.5, NZ - 0.5], [-0.5, NZ - 0.5], color="#AEB9C5", linestyle=":", linewidth=2.43, alpha=0.35, zorder=6)
        ax.axvline(um, color="#AEB9C5", linestyle=":", linewidth=2.43, alpha=0.35, zorder=6)
        ax.axhline(um, color="#AEB9C5", linestyle=":", linewidth=2.43, alpha=0.35, zorder=6)
        desenhar_fronteira(ax, mapa > 0, FRONT_OBS)
        handles = [Line2D([0], [0], label=virgula(rf"Fronteira observada: $\delta={dd:g}$"), **FRONT_OBS)]
        for k, (dr, estilo) in enumerate(DELTAS_REF):
            s_ref, d_ref = na_grade(d, dr)
            desenhar_fronteira(ax, mapa_binado(s_ref, "N_malha") > 0, estilo, zorder=18 + k)
            handles.append(Line2D([0], [0], label=virgula(rf"Fronteira de referência: $\delta={d_ref:g}$"), **estilo))
        fronteira_logistica(ax, dd, f)
        handles.append(Line2D([0], [0], label="Fronteira logística\n" + r"($P_{\mathrm{res}}=0{,}5$)", **FRONT_LOG))
        eixos_alpha_beta(ax, *ROTULOS_EIXOS[f])
        legenda_mapa(ax, handles)
        ax.text(0.5, 0.965, virgula(rf"$\delta={dd:g}$"), transform=ax.transAxes, fontsize=24, color="white", va="top", ha="center",
                bbox=dict(facecolor=BOX_COLOR, edgecolor="none", alpha=1.0, boxstyle="square,pad=0.16"), zorder=110)
        caixa_mapa(ax, r"$N_{\mathrm{res}}$")
        ticks = ticks_ressonancias(n_max)
        cbar = barra_mapa(fig, cax, norm, np.log10(1.0 + ticks), [str(t) for t in ticks], "Ressonâncias")
        acerto = np.mean((p_res(sub["alpha"], sub["beta"], sub["delta"], f, COEF_EXIST) >= 0.5) == sub["ressonante"])
        print(f"f = {f:g}, delta = {dd:g}: {len(sub)} curvas, N máximo = {n_max}, P_res >= 0,5 concorda com a contagem final em {100 * acerto:.1f} %")
        concluir_mapa(fig, ax, cbar, f"mapa_estados_f{f:.2f}_delta{dd:g}")

## 8. Relação de recorrência

### 8.1 Esfera homogênea: zeros de $j_{\ell-1}$

Para a esfera fluida $A$ de raio $a$ em $M$, a ressonância de ordem $\ell$ satisfaz (Eq. 4.20)
$\dfrac{\rho_M}{\rho_A}\dfrac{x_Aj_\ell'(x_A)}{j_\ell(x_A)}=\dfrac{x_Mh_\ell^{(1)\prime}(x_M)}{h_\ell^{(1)}(x_M)}$, com $x_A=m\,x_M$ e $m=c_M/c_A$.
Para $x_M\ll1$ o lado direito tende a $-(\ell+1)$ e a condição fica (Eq. 4.21)

$$
\frac{\rho_M}{\rho_A}\,x_Aj_\ell'(x_A)+(\ell+1)\,j_\ell(x_A)=0 .
$$

Para $\ell=0$ e $\rho_A\ll\rho_M$ ela dá a escala de Minnaert, $x_{\mathrm{ref}}=\sqrt{3/(m\,m_t)}$ com $m_t=Z_M/Z_A$ (Eq. 4.23). Com densidades iguais,
$x\,j_\ell'+(\ell+1)j_\ell=x\,j_{\ell-1}$ e as ressonâncias são os zeros de $j_{\ell-1}$ (Eqs. 4.25 e 4.26):

$$
x_M=\frac{z_{\ell-1,n}}{m},\qquad j_{-1}(x)=\frac{\cos x}{x},\qquad z_{\ell-1,n+1}-z_{\ell-1,n}\longrightarrow\pi .
$$

### 8.2 Core-shell: espaçamento médio em função de $f$

A fase acumulada no interior da partícula é $\Theta=x_M\,(c_M/c_A+\delta\,c_M/c_B)=\Phi\,x_M$ (Eq. 4.33). Cada aumento de $\pi$ em $\Theta$ traz uma raiz
nova, de modo que $N(x_M)=\Phi x_M/\pi+O(1)$ (Eq. 4.35) e o espaçamento médio é (Eqs. 4.36 e 4.37)

$$
\Delta_W=\frac{\pi}{c_M/c_A+\delta\,c_M/c_B}=\frac{\pi}{\alpha^{-(1-f)}+\delta\,\beta^{-(1-f)}}=\pi\,e^{-u(\alpha,\beta,\delta,f)},
$$

com a mesma variável $u$ da relação de existência. Com $A=B$, $\Delta_W=\pi/[m(1+\delta)]$ (Eq. 4.41), que em unidades do raio externo é o passo $\pi/m$
da esfera homogênea; com $\delta\to0$, $\Delta_W\to\pi/m_A$, o passo da esfera $A$; e em $f=1$, $\Delta_W=\pi/(1+\delta)$ (Eq. 4.38).

### 8.3 Recorrência e o fator $S(f)$

Trocando, no passo da esfera homogênea $(z_{\ell-1,n+1}-z_{\ell-1,n})/m$, o fator $\pi/m$ por $\Delta_W$ (Eq. 4.43):

$$
x^{(\ell)}_{M,n+1}=x^{(\ell)}_{M,n}+S(f)\,e^{-u}\,\big(z_{\ell-1,n+1}-z_{\ell-1,n}\big)
\quad\Longleftrightarrow\quad
x^{(\ell)}_{M,n}=x^{(\ell)}_{M,1}+S(f)\,e^{-u}\,\big(z_{\ell-1,n}-z_{\ell-1,1}\big).
$$

**Forma de $S(f)$.** Em $f=0$ as densidades são iguais e a recorrência é exata com $S=1$ (Eq. 4.25), logo $S(0)=1$. Para $f>0$, as raízes da
Eq. (4.21) passam a depender das razões de densidade: com as formas assintóticas das funções de Bessel, a raiz de $x\,j_\ell'+(\ell+1)\,r\,j_\ell=0$ fica
$\zeta\simeq z_{\ell-1,s}+\arctan\big[(r-1)(\ell+1)/z_{\ell-1,s}\big]$, e o passo entre raízes vizinhas é corrigido por potências de $r-1\simeq\ln r$. Na
parametrização do capítulo, todas as razões de densidade são potências de $f$:

$$
\ln\frac{\rho_A}{\rho_B}=f\ln\frac{\alpha}{\beta},\qquad\ln\frac{\rho_B}{\rho_M}=f\ln\beta,
$$

de modo que a correção de primeira ordem é proporcional a $f\ln(\alpha/\beta)$ e a $f\ln\beta$. Esse termo é ímpar na troca $Z\to1/Z$ e se cancela na mediana
sobre as famílias, porque as grades cobrem $\ln\alpha$ e $\ln\beta$ de forma praticamente simétrica em torno de zero e balanceada em $\eta$. O primeiro
termo que sobrevive é quadrático, $(f\ln Z)^2$, cuja média sobre o banco é uma constante vezes $f^2$. Assim,

$$
S(f)=1+s\,f^2 ,
$$

com um único coeficiente, $s$, **ajustado aqui** a todas as famílias de treino, de todos os $f$. Para cada família $(\alpha,\beta,\delta,f,\ell)$ com pelo
menos duas ressonâncias, $S_{\mathrm{fam}}$ sai de mínimos quadrados na forma da direita ($n\ge2$), como antes. Como $S_{\mathrm{fam}}-1=s\,f^2$, o ajuste
de $s$ pela mediana é a mediana ponderada de $(S_{\mathrm{fam}}-1)/f^2$ com pesos $f^2$ (a solução de mínimos desvios absolutos da forma acima); o erro da
mediana vem de $300$ reamostragens das famílias (semente 42).

### 8.4 Primeiro termo de cada família, em função de $f$

Sem busca de raiz. Quando $\Phi x_{M,1}<1$, as funções de Bessel do denominador quase-estático são trocadas pelas expansões de pequeno argumento
(Eq. 4.44) e a condição de ressonância, mantida até $x_M^2$, fica algébrica: $\widetilde D_\ell\simeq a_0+\big(m_A^2\,p_A+m_B^2\,p_B\big)x_M^2$, de modo que
$x_{M,1}^2=-a_0/(m_A^2p_A+m_B^2p_B)$, com $a_0$, $p_A$ e $p_B$ dependentes só de $\ell$, $\delta$ e das razões de densidade. Em função de $f$,
$m_A=c_M/c_A=\alpha^{-(1-f)}$, $m_B=\beta^{-(1-f)}$, $\rho_A/\rho_M=\alpha^f$ e $\rho_B/\rho_M=\beta^f$. Para $\ell=0$, com $O=1+\delta$, isso é a Eq. (4.45):

$$
x_{M,1}^2=\frac{6\,O\,\alpha^f\beta^f}{\alpha^{-2(1-f)}\,\beta^f\big[O(\alpha^f+2\beta^f)-2\beta^f+2\big]+\beta^{-2(1-f)}\,\alpha^f\big[O^3(\beta^f+2)-3O\beta^f+2\beta^f-2\big]} .
$$

Fora desse limite (ou para $\ell>2$), a primeira ressonância é a da cavidade de menor frequência, o núcleo (parede: a casca) ou a partícula inteira
(parede: o meio externo) (Eq. 4.46), que em função de $f$ fica

$$
x^{(\ell)}_{M,1}=\min\left\{\frac{w_\ell(\rho_A/\rho_B)}{c_M/c_A},\ \frac{w_\ell(\rho_B/\rho_M)}{(1+\delta)\,c_M/c_B}\right\}
=\min\left\{\alpha^{1-f}\,w_\ell\!\left[(\alpha/\beta)^f\right],\ \frac{\beta^{1-f}\,w_\ell\!\left(\beta^f\right)}{1+\delta}\right\},
$$

com $w_\ell(r)$ o primeiro zero positivo de $x\,j_\ell'(x)+(\ell+1)\,r\,j_\ell(x)$ (zeros de $j_\ell'$ para $r\to0$, de $j_{\ell-1}$ para $r=1$ e de
$j_\ell$ para $r\to\infty$). Nenhum coeficiente é ajustado nesta etapa. A célula seguinte confere que $\Delta_W=\pi e^{-u}$, que a forma geral para
$\ell=0$ coincide com a Eq. (4.45) escrita em $f$ e que $w_\ell(r)$ tem os limites esperados.

In [ ]:
_ZEROS, _W = {}, {}


def _jl(l, x):
    return np.cos(x) / x if l == -1 else spherical_jn(l, x)


def zeros_j(l, n):
    # z_{l,1}, ..., z_{l,n}: zeros positivos de j_l (l = -1: cos x/x), calculados uma vez e guardados
    if l in _ZEROS and len(_ZEROS[l]) >= n:
        return _ZEROS[l][:n]
    x = np.arange(max(l, 0) * 0.5 + 1e-6, (n + l + 3) * np.pi, 0.02)
    v = _jl(l, x)
    i = np.flatnonzero(np.sign(v[1:]) != np.sign(v[:-1]))[:n]
    a, b = x[i], x[i + 1]
    for _ in range(55):
        m = 0.5 * (a + b)
        e = np.sign(_jl(l, m)) == np.sign(_jl(l, a))
        a, b = np.where(e, m, a), np.where(e, b, m)
    _ZEROS[l] = 0.5 * (a + b)
    return _ZEROS[l]


def _robin(x, l, r):
    return x * spherical_jn(l, x, derivative=True) + (l + 1) * r * spherical_jn(l, x)


def w_l(l, r):
    # primeiro zero positivo de x j_l'(x) + (l+1) r j_l(x) (tabelado: depende só de l e de r)
    chave = (l, round(float(np.log(r)), 10))
    if chave not in _W:
        x = np.geomspace(1e-9, zeros_j(l, 1)[0] * (1 + 1e-9), 600)
        v = _robin(x, l, r)
        i = np.flatnonzero(np.sign(v[1:]) != np.sign(v[:-1]))
        _W[chave] = brentq(_robin, x[i[0]], x[i[0] + 1], args=(l, r), xtol=1e-14) if len(i) else np.nan
    return _W[chave]


def coef_baixa_frequencia(l, delta, rA, rB, rM):
    # a_0 e (p_A, p_B) da condição quase-estática D_l ~ a_0 + (m_A^2 p_A + m_B^2 p_B) x_M^2, para l = 0, 1, 2
    rAB, rMB, O = rA / rB, rM / rB, 1 + delta
    T1 = (l * (1 - rAB), np.array([l * rAB - l - 2, (l + 2) * rAB - l]) / (2 * (2 * l + 3)))
    T2 = ((l + 1) * (1 - rMB), np.array([0.0, O ** 2 * (l + 1 - (l - 1) * rMB) / (2 * (2 * l - 1))]))
    T3 = (O ** (2 * l + 1) * (l + (l + 1) * rAB), O ** (2 * l + 1) * np.array([-(l + 2 + (l + 1) * rAB) / (2 * (2 * l + 3)), (l + (l - 1) * rAB) / (2 * (2 * l - 1))]))
    T4 = (l + 1 + l * rMB, np.array([0.0, -O ** 2 * (l + 1 + (l + 2) * rMB) / (2 * (2 * l + 3))]))
    return T1[0] * T2[0] - T3[0] * T4[0], T1[0] * T2[1] + T1[1] * T2[0] - T3[0] * T4[1] - T3[1] * T4[0]


def primeira_ressonancia(l, delta, mat):
    # x_{M,1} da família l: limite quase-estático (l <= 2, se Phi x_{M,1} < 1) ou cavidade de menor frequência
    rM, rA, rB, cM, cA, cB = mat
    mA, mB = cM / cA, cM / cB
    if l <= 2:
        a0, (pA, pB) = coef_baixa_frequencia(l, delta, rA, rB, rM)
        x2 = -a0 / (mA ** 2 * pA + mB ** 2 * pB)
        if x2 > 0 and (mA + delta * mB) * np.sqrt(x2) < 1:
            return float(np.sqrt(x2)), "baixa frequência"
    return float(min(w_l(l, rA / rB) / mA, w_l(l, rB / rM) / ((1 + delta) * mB))), "cavidade"


def delta_W(delta, mat):
    rM, rA, rB, cM, cA, cB = mat
    return np.pi / (cM / cA + delta * cM / cB)


def familia(l, delta, mat, S, x_max=X_MAX, x1=None):
    # x_{M,n}^{(l)} = x_{M,1} + S (Delta_W/pi) (z_{l-1,n} - z_{l-1,1}) até x_max
    if x1 is None:
        x1 = primeira_ressonancia(l, delta, mat)[0]
    passo = S * delta_W(delta, mat) / np.pi
    if not (np.isfinite(x1) and x1 <= x_max and np.isfinite(passo) and passo > 0):
        return np.array([])
    z = zeros_j(l - 1, int((x_max - x1) / (max(passo, 1e-300) * 2.5)) + 3)
    x = x1 + passo * (z - z[0])
    return x[x <= x_max]


rng = np.random.default_rng(1)
a, b, d, f = 10 ** rng.uniform(-3, 3.4, 200), 10 ** rng.uniform(-3, 3.4, 200), rng.uniform(0.01, 1.5, 200), rng.uniform(0, 1, 200)
print(f"Delta_W = pi e^-u: maior diferença relativa {np.max(np.abs(delta_W(d, materiais(a, b, f)) / (np.pi * np.exp(-variavel_u(a, b, d, f))) - 1)):.1e}")
x2_geral = np.array([(lambda c: -c[0] / ((C_AGUA / (C_AGUA * ai ** (1 - fi))) ** 2 * c[1][0] + (C_AGUA / (C_AGUA * bi ** (1 - fi))) ** 2 * c[1][1]))(
    coef_baixa_frequencia(0, di, ai ** fi, bi ** fi, 1.0)) for ai, bi, di, fi in zip(a, b, d, f)])
x1_445 = x1_monopolo(a, b, d, f)
ok = np.isfinite(x1_445)
print(f"l = 0: forma geral contra a Eq. (4.45) em f, maior diferença relativa {np.max(np.abs(np.sqrt(x2_geral[ok]) / x1_445[ok] - 1)):.1e} ({ok.sum()} configurações)")
for l in (0, 1, 2):
    print(f"w_{l}: r = 1e-8 -> {w_l(l, 1e-8):.5f} | r = 1 -> {w_l(l, 1.0):.5f} (z_{{l-1,1}} = {zeros_j(l - 1, 1)[0]:.5f}) | "
          f"r = 1e8 -> {w_l(l, 1e8):.5f} (z_{{l,1}} = {zeros_j(l, 1)[0]:.5f})")

### 8.5 Ajuste de $S(f)$

Famílias observadas nas curvas de treino (as mesmas $80\,\%$ da relação de existência), de todas as grades e das configurações sorteadas.
A figura mostra $S_{\mathrm{fam}}$ de cada família contra $f$, a mediana das famílias em faixas de $f$ (só para conferir a forma) e a curva ajustada.

In [ ]:
def familias_observadas(res, minimo=2):
    r = res.loc[res["primary_l"].notna()].copy()
    r["l"] = r["primary_l"].astype(int)
    r = r.sort_values(["curva", "l", "xM_peak"])
    r["n"] = r.groupby(["curva", "l"]).cumcount() + 1
    r["N_fam"] = r.groupby(["curva", "l"])["xM_peak"].transform("size")
    r = r.loc[r["N_fam"] >= minimo].copy()
    mat = materiais(r["alpha"].to_numpy(), r["beta"].to_numpy(), r["f"].to_numpy())
    r["Phi"] = mat[3] / mat[4] + r["delta"] * mat[3] / mat[5]
    r["DW"] = np.pi / r["Phi"]
    z = np.full(len(r), np.nan)
    for l in np.unique(r["l"]):
        k = (r["l"] == l).to_numpy()
        z[k] = zeros_j(l - 1, int(r.loc[k, "n"].max()))[r.loc[k, "n"].to_numpy() - 1]
    r["z"] = z
    r["z1"] = r.groupby(["curva", "l"])["z"].transform("first")
    r["x1"] = r.groupby(["curva", "l"])["xM_peak"].transform("first")
    return r


def S_por_familia(fam):
    # S de cada família por mínimos quadrados em x_n - x_1 = S (Delta_W/pi)(z_n - z_1), n >= 2
    d = fam.loc[fam["n"] >= 2].assign(o=lambda t: t["xM_peak"] - t["x1"], p=lambda t: t["DW"] / np.pi * (t["z"] - t["z1"]))
    g = d.assign(op=d["o"] * d["p"], pp=d["p"] ** 2).groupby(["curva", "l"]).agg(op=("op", "sum"), pp=("pp", "sum"), f=("f", "first"), N=("n", "max"))
    g["S"] = g["op"] / g["pp"]
    return g.loc[np.isfinite(g["S"])].reset_index()


def mediana_ponderada(v, w):
    o = np.argsort(v)
    c = np.cumsum(w[o])
    return float(v[o][np.searchsorted(c, 0.5 * c[-1])])


def ajusta_S(tab, n_boot=300):
    # S(f) = 1 + s f^2: s = mediana de (S_fam - 1)/f^2 com pesos f^2; erro por reamostragem das famílias
    k = tab["f"].to_numpy() > 0
    w = tab["f"].to_numpy()[k] ** 2
    v = (tab["S"].to_numpy()[k] - 1) / w
    if not len(v):
        return np.nan, np.nan, 0
    rng = np.random.default_rng(RANDOM_SEED)
    boot = [mediana_ponderada(v[i], w[i]) for i in (rng.integers(0, len(v), len(v)) for _ in range(n_boot))]
    return mediana_ponderada(v, w), float(np.std(boot)), int(len(v))


def S_de_f(f):
    return 1.0 + S2 * np.asarray(f, float) ** 2


FAM = familias_observadas(RES)
TAB_S = S_por_familia(FAM.loc[~FAM["curva"].isin(CURVAS_TESTE)])
S2, ERRO_S2, N_FAM_S = ajusta_S(TAB_S)
print(f"S(f) = 1 + s f^2: s = {S2:.5f} ± {ERRO_S2:.5f} ({N_FAM_S} famílias de treino com f > 0; {int((TAB_S['f'] == 0).sum())} em f = 0)")
if (TAB_S["f"] == 0).any():
    s0 = TAB_S.loc[TAB_S["f"] == 0, "S"]
    print(f"em f = 0 (sem coeficiente): mediana de S_fam = {s0.median():.5f} ± {1.2533 * s0.std() / np.sqrt(len(s0)):.5f}")

i_f, centros = faixas_f(TAB_S)
fx = pd.DataFrame({"f": centros[i_f], "S": TAB_S["S"]}).groupby("f")["S"].agg(["median", "std", "size"])
fx = fx.loc[fx["size"] >= 3]
fig, ax = plt.subplots(figsize=(5, 3.5))
ax.scatter(TAB_S["f"], TAB_S["S"], s=10, facecolors="white", edgecolors="0.45", linewidths=0.5, zorder=1, rasterized=True, label="Famílias de treino")
ax.errorbar(fx.index, fx["median"], yerr=1.2533 * fx["std"] / np.sqrt(fx["size"]), fmt="o", markersize=4, color=COLORS[0], ecolor=COLORS[0],
            capsize=2, linewidth=1.2, zorder=3, label="Mediana por faixa de $f$")
fg = np.linspace(0, 1, 200)
ax.plot(fg, S_de_f(fg), color=COLORS[1], linestyle=LINESTYLES[1], linewidth=LINEWIDTHS[1], zorder=4, label=rf"$S(f)=1{'+' if S2 >= 0 else '-'}{dec(abs(S2), 3)}\,f^2$")
lo, hi = np.nanpercentile(TAB_S["S"], [2, 98]) if len(TAB_S) else (0.8, 1.4)
ax.set_ylim(min(lo, 0.9), max(hi, S_de_f(1.0) * 1.05))
eixo_x_fixo(ax, [0.0, 0.25, 0.5, 0.75, 1.0], (-0.02, 1.02))
ax.yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"${dec(v, 2)}$"))
estilizar(ax, r"$f$", r"$S$")
adicionar_legenda(ax, loc="upper left")
fig.subplots_adjust(left=0.14, right=0.985, bottom=0.15, top=0.98)
concluir(fig, "grafico_S_contra_f")

### 8.6 Espaçamento médio medido

Para cada família com pelo menos seis ressonâncias em $0<x_M\le1$ (todas as curvas, todos os $f$), mede-se o espaçamento médio
$\overline{\Delta x}_M$ na metade superior da família e compara-se $\pi/\overline{\Delta x}_M$ com $\Phi=c_M/c_A+\delta\,c_M/c_B=e^u$ (a linha tracejada é
$\overline{\Delta x}_M=\Delta_W$, Eq. 4.37). O ajuste livre $\pi/\overline{\Delta x}_M=p_1\,c_M/c_A+p_2\,\delta\,c_M/c_B+p_0$, um só para todo o banco, mostra se os
coeficientes $1$ da Eq. (4.37) são recuperados pelos próprios dados. A cor é o número de famílias em cada célula.

In [ ]:
def media_superior(x):
    x = np.sort(x)
    sup = x[len(x) // 2:]
    return np.mean(np.diff(sup)) if len(sup) >= 2 else np.nan


g = FAM.loc[FAM["N_fam"] >= 6].groupby(["curva", "l"]).agg(Phi=("Phi", "first"), alpha=("alpha", "first"), beta=("beta", "first"),
                                                            delta=("delta", "first"), f=("f", "first"), dx=("xM_peak", media_superior))
g = g.loc[np.isfinite(g["dx"]) & (g["dx"] > 0)]
ESPACAMENTO = {}
if len(g) >= 3:
    yv, xv = np.pi / g["dx"].to_numpy(), g["Phi"].to_numpy()
    mat = materiais(g["alpha"].to_numpy(), g["beta"].to_numpy(), g["f"].to_numpy())
    A = np.column_stack([mat[3] / mat[4], g["delta"] * mat[3] / mat[5], np.ones(len(g))])
    P_ESP = np.linalg.lstsq(A, yv, rcond=None)[0]
    ESPACAMENTO = dict(familias=len(g), p1=P_ESP[0], p2=P_ESP[1], p0=P_ESP[2], desvio_ajuste=100 * np.median(np.abs(A @ P_ESP - yv) / yv),
                       desvio_eq_4_37=100 * np.median(np.abs(xv - yv) / yv))
    print(f"{len(g)} famílias | p1 = {P_ESP[0]:.4f}, p2 = {P_ESP[1]:.4f}, p0 = {P_ESP[2]:+.3f} | desvio mediano do ajuste {ESPACAMENTO['desvio_ajuste']:.2f} % "
          f"| da Eq. (4.37), sem coeficientes, {ESPACAMENTO['desvio_eq_4_37']:.2f} %")
    lo, hi = 10 ** np.floor(np.log10(min(xv.min(), yv.min()))), 10 ** np.ceil(np.log10(max(xv.max(), yv.max())))
    with plt.rc_context(MAPA_RC):
        fig, ax, cax = figura_mapa()
        hb = ax.hexbin(xv, yv, xscale="log", yscale="log", gridsize=30, mincnt=1, cmap=RESONANCE_CMAP, linewidths=CELL_LINEWIDTH, edgecolors=CELL_LINECOLOR,
                       extent=(np.log10(lo), np.log10(hi), np.log10(lo), np.log10(hi)))
        cont = np.asarray(hb.get_array(), float)
        hb.set_norm(LogNorm(vmin=0.5, vmax=max(cont.max(), 2)))
        ax.plot([lo, hi], [lo, hi], color="black", linestyle=(0, (5.0, 2.2)), linewidth=2.0, zorder=5, label=r"$\pi/\overline{\Delta x}_M=c_M/c_A+\delta\,c_M/c_B$")
        ax.set_xlim(lo, hi); ax.set_ylim(lo, hi)
        ax.set_aspect("equal", adjustable="box")
        for eixo in (ax.xaxis, ax.yaxis):
            eixo.set_major_formatter(FuncFormatter(lambda v, _: f"{v:g}".replace(".", ",")))
        ax.set_xlabel(r"$c_M/c_A+\delta\,c_M/c_B$", fontsize=23, labelpad=9)
        ax.set_ylabel(r"$\pi/\overline{\Delta x}_M$", fontsize=23, labelpad=10)
        legenda_mapa(ax, [Line2D([0], [0], color="black", linestyle=(0, (5.0, 2.2)), linewidth=2.0, label=r"$\pi/\overline{\Delta x}_M=c_M/c_A+\delta\,c_M/c_B$")],
                     loc="lower right", bbox=(0.985, 0.015))
        t = [v for v in (1, 2, 5, 10, 20, 50, 100, 200, 500, 1000) if v <= cont.max()]
        cbar = fig.colorbar(hb, cax=cax, ticks=t)
        cbar.ax.set_yticklabels([str(v) for v in t])
        cbar.minorticks_off()
        cbar.set_label("Famílias", fontsize=20, labelpad=18, rotation=270)
        concluir_mapa(fig, ax, cbar, "espacamento_medio")
else:
    print("menos de três famílias com seis ressonâncias: sem gráfico do espaçamento")

### 8.7 Validação nas configurações de teste

As mesmas configurações de teste da relação de existência (nenhuma delas entrou no ajuste de $S$). Para cada família observada: a primeira
ressonância pelas Eqs. (4.45)/(4.46), as posições a partir da segunda pela Eq. (4.43) partindo do primeiro termo previsto e o número de
ressonâncias até $x_M=1$ (somando as famílias com $\ell$ observado na curva).

**Recorrência contra o dataset.** Para cada par de ressonâncias consecutivas $(x_{M,n},x_{M,n+1})$ de $W_T$ nas curvas de teste, a posição seguinte
prevista pela recorrência vem, para cada ordem $\ell$ presente na curva, da última ressonância dessa ordem observada até $x_{M,n}$ (ou do primeiro
termo, se a ordem ainda não apareceu), avançando pela Eq. (4.43) até passar de $x_{M,n}$; a ressonância seguinte é a menor entre as ordens:

$$
x^{\mathrm{recorr\hat encia}}_{M,n+1}=\min_{\ell}\Big\{x^{(\ell)}_{M,k}+S(f)\,e^{-u}\big(z_{\ell-1,k+j}-z_{\ell-1,k}\big)\ :\ x^{(\ell)}_{M,k+j}>x_{M,n}\Big\}.
$$

O hexbin compara o passo previsto, $x^{\mathrm{recorr\hat encia}}_{M,n+1}-x_{M,n}$, com o observado, e a cor é o erro relativo mediano na posição
$x_{M,n+1}$ em cada célula. Os erros também são mostrados em função de $f$.

In [ ]:
RES_TESTE = RES.loc[RES["curva"].isin(CURVAS_TESTE)]
FAM_TESTE = familias_observadas(RES_TESTE, minimo=1)
linhas, contagem = [], []
for (curva, l), t in FAM_TESTE.groupby(["curva", "l"]):
    a, b, d, f = t.iloc[0][["alpha", "beta", "delta", "f"]]
    mat = materiais(a, b, f)
    x1, origem = primeira_ressonancia(int(l), d, mat)
    prev = familia(int(l), d, mat, S_de_f(f), x_max=10.0, x1=x1)[: len(t)]
    for n, (xp, xo) in enumerate(zip(prev, t["xM_peak"].to_numpy()), 1):
        linhas.append((curva, int(l), f, n, xo, xp, origem))
VAL = pd.DataFrame(linhas, columns=["curva", "l", "f", "n", "obs", "prev", "origem"])
VAL["erro"] = 100 * np.abs(VAL["prev"] - VAL["obs"]) / VAL["obs"]
ls_curva = FAM_TESTE.groupby("curva")["l"].unique()
for _, c in teste_c.loc[teste_c["N"] > 0].iterrows():
    if c["curva"] in ls_curva.index:
        mat = materiais(c["alpha"], c["beta"], c["f"])
        contagem.append((c["curva"], c["f"], c["N"], sum(len(familia(int(l), c["delta"], mat, S_de_f(c["f"]))) for l in ls_curva[c["curva"]])))
CONT = pd.DataFrame(contagem, columns=["curva", "f", "obs", "prev"])
CONT["erro"] = 100 * np.abs(CONT["prev"] - CONT["obs"]) / CONT["obs"]


def proximas_previstas(x, l, delta, mat, S):
    # x: picos de W_T da curva em ordem, l: ordem de cada pico; devolve a previsão de x_{n+1} para n = 1, ..., N-1
    passo = S * delta_W(delta, mat) / np.pi
    xn = x[:-1]
    prev = np.full(len(xn), np.inf)
    for L in np.unique(l):
        e_L = l == L
        nL = np.cumsum(e_L)[:-1]                                   # ressonâncias de ordem L até x_n
        x1 = primeira_ressonancia(int(L), delta, mat)[0]
        base = np.where(nL > 0, x[e_L][np.maximum(nL - 1, 0)], x1)
        n0 = np.maximum(nL, 1)
        z = zeros_j(int(L) - 1, int(n0.max() + (x[-1] - base.min()) / (passo * 2.5)) + 10)
        alvo = z[n0 - 1] + (xn - base) / passo
        k = np.minimum(np.searchsorted(z, alvo * (1 + 1e-12) + 1e-12, side="right"), len(z) - 1)
        cand = base + passo * (z[k] - z[n0 - 1])
        cand = np.where((nL == 0) & (x1 > xn), x1, cand)
        prev = np.minimum(prev, np.where(cand > xn, cand, np.inf))
    return np.where(np.isfinite(prev), prev, np.nan)


def pares_consecutivos(res):
    partes = []
    for curva, t in res.groupby("curva"):
        if len(t) < 2 or t["primary_l"].isna().any():
            continue
        t = t.sort_values("xM_peak")
        a, b, d, f = t.iloc[0][["alpha", "beta", "delta", "f"]]
        x = t["xM_peak"].to_numpy(float)
        partes.append(pd.DataFrame({"curva": curva, "f": f, "x_n": x[:-1], "x_np1": x[1:],
                                    "x_prev": proximas_previstas(x, t["primary_l"].to_numpy(int), d, materiais(a, b, f), S_de_f(f))}))
    return pd.concat(partes, ignore_index=True) if partes else pd.DataFrame(columns=["curva", "f", "x_n", "x_np1", "x_prev"])


base_prova = MALHA if len(MALHA) and MALHA["primary_l"].notna().any() else RES
PARES = pares_consecutivos(base_prova.loc[base_prova["curva"].isin(CURVAS_TESTE)])
PARES["erro"] = 100 * np.abs(PARES["x_prev"] - PARES["x_np1"]) / PARES["x_np1"]
ok = np.isfinite(PARES["x_prev"]) & (PARES["x_prev"] > PARES["x_n"]) & (PARES["x_np1"] > PARES["x_n"])
PROVA = dict(pares=int(ok.sum()), mediana=float(PARES.loc[ok, "erro"].median()), p90=float(PARES.loc[ok, "erro"].quantile(0.9)), maximo=float(PARES.loc[ok, "erro"].max()))
VALIDACAO = dict(primeira=float(VAL.loc[VAL["n"] == 1, "erro"].median()), posicoes_n2=float(VAL.loc[VAL["n"] >= 2, "erro"].median()),
                 contagem=float(CONT["erro"].median()), familias=int((VAL["n"] == 1).sum()), curvas=len(CONT), **{f"recorrencia_{k}": v for k, v in PROVA.items()})
print(f"erro relativo mediano: primeira ressonância {VALIDACAO['primeira']:.3g} % ({VALIDACAO['familias']} famílias) | posições, n >= 2, "
      f"{VALIDACAO['posicoes_n2']:.3g} % | contagem até x_M = 1 {VALIDACAO['contagem']:.3g} % ({len(CONT)} curvas)")
print(f"recorrência contra o dataset: {PROVA['pares']} pares | erro relativo na posição: mediana {PROVA['mediana']:.3g} %, "
      f"90 % abaixo de {PROVA['p90']:.3g} %, máximo {PROVA['maximo']:.3g} %")
print("\nprimeira ressonância, erro mediano por l (%):\n" + VAL.loc[VAL["n"] == 1].groupby("l")["erro"].median().round(2).to_string())
VAL.to_csv(os.path.join(TAB_DIR, "teste_posicoes.csv"), index=False)
CONT.to_csv(os.path.join(TAB_DIR, "teste_contagem.csv"), index=False)
PARES.to_csv(os.path.join(TAB_DIR, "pares_recorrencia_teste.csv"), index=False)

if ok.sum() >= 3:
    LO, HI = 1e-5, 1.0
    passo_prev, passo_obs, erro = (PARES.loc[ok, "x_prev"] - PARES.loc[ok, "x_n"]).to_numpy(), (PARES.loc[ok, "x_np1"] - PARES.loc[ok, "x_n"]).to_numpy(), PARES.loc[ok, "erro"].to_numpy()
    fig, ax = plt.subplots(figsize=(5, 3.5))
    hb = ax.hexbin(passo_prev, passo_obs, C=erro, reduce_C_function=np.median, xscale="log", yscale="log", gridsize=45, cmap=RESONANCE_CMAP, mincnt=1,
                   linewidths=0.15, edgecolors="white", extent=(np.log10(LO), np.log10(HI), np.log10(LO), np.log10(HI)))
    cel = np.asarray(hb.get_array(), float)
    cel = cel[np.isfinite(cel) & (cel > 0)]
    e_min = 10.0 ** np.floor(np.log10(cel.min())) if len(cel) else 1e-3
    e_max = max(cel.max() if len(cel) else 1.0, 10 * e_min)
    hb.set_norm(LogNorm(vmin=e_min / 3, vmax=e_max, clip=True))           # a menor célula fica clara, não branca
    ticks_cor = 10.0 ** np.arange(np.log10(e_min), np.floor(np.log10(e_max)) + 1)
    ticks_cor = ticks_cor[::-1][::int(np.ceil((len(ticks_cor) - 1) / 3)) or 1][::-1]
    ax.plot([LO, HI], [LO, HI], color="black", linestyle=(0, (5.0, 2.2)), linewidth=1.0, zorder=5,
            label=r"$x^{\mathrm{recorr\hat{e}ncia}}_{M,n+1}=x^{\mathrm{dataset}}_{M,n+1}$")
    ax.set_xlim(LO, HI); ax.set_ylim(LO, HI)
    ax.set_aspect("equal", adjustable="box")
    for eixo in (ax.xaxis, ax.yaxis):
        eixo.set_major_locator(FixedLocator([1e-4, 1e-2, 1.0]))
        eixo.set_minor_locator(NullLocator())
        eixo.set_major_formatter(FuncFormatter(potencia))
    estilizar(ax, r"$x^{\mathrm{recorr\hat{e}ncia}}_{M,n+1}-x_{M,n}$", r"$x^{\mathrm{dataset}}_{M,n+1}-x_{M,n}$")
    adicionar_legenda(ax, loc="upper left")
    cbar = fig.colorbar(hb, ax=ax, ticks=ticks_cor, pad=0.025, fraction=0.046, aspect=24)
    cbar.ax.yaxis.set_major_formatter(FuncFormatter(potencia))
    cbar.set_label(f"Erro relativo na posição ({PCT})", fontsize=12, labelpad=15, rotation=-90)
    cbar.ax.tick_params(labelsize=10, width=0.9, length=4, direction="in")
    cbar.minorticks_off()
    cbar.outline.set_edgecolor("black"); cbar.outline.set_linewidth(1.0)
    concluir(fig, "grafico_recorrencia")

fig, ax = plt.subplots(figsize=(5, 3.5))
for i, (df, mk, rot) in enumerate(((VAL, VAL["n"] == 1, "Primeira ressonância"), (VAL, VAL["n"] >= 2, r"Posições, $n\geq2$"),
                                    (PARES, ok, r"Próxima ressonância de $W_T$"), (CONT, np.ones(len(CONT), bool), r"Contagem até $x_M=1$"))):
    d = df.loc[mk]
    if not len(d):
        continue
    k, c = faixas_f(d)
    med = pd.Series(d["erro"].to_numpy(), index=c[k]).groupby(level=0).median()
    ax.plot(med.index, med.to_numpy(), color=COLORS[i], linestyle=LINESTYLES[i], linewidth=LINEWIDTHS[i], marker="o", markersize=3, label=rot)
ax.set_yscale("log")
formatar_log(ax)
eixo_x_fixo(ax, [0.0, 0.25, 0.5, 0.75, 1.0])
estilizar(ax, r"$f$", f"Erro relativo mediano ({PCT})")
adicionar_legenda(ax, loc="upper left", fontsize=8)
fig.subplots_adjust(left=0.14, right=0.985, bottom=0.15, top=0.98)
concluir(fig, "grafico_erro_recorrencia_contra_f")

## 9. Tabela de coeficientes

Uma única tabela reúne os coeficientes ajustados das duas relações, com valor $\pm$ incerteza (dois algarismos significativos na incerteza): os
$c_i$ da relação de existência, Eq. (4.19), e o coeficiente $s$ de $S(f)=1+s\,f^2$ da relação de recorrência, Eq. (4.43). Sai em PNG, PDF, SVG, CSV e
`.tex`; os coeficientes e as métricas também vão para `tabelas/relacoes_ajustadas.json`.

In [ ]:
TERMOS_TAB = [t.replace("$", "") for t in TEX_TERMOS]
linhas = [f"$c_{{{i}}}$" for i in range(10)] + ["$s$"]
termos = TEX_TERMOS + [r"$S(f)=1+s\,f^2$"]
valores, erros = np.r_[COEF_EXIST, S2], np.r_[ERRO_EXIST, ERRO_S2]
salvar_tabela("tabela_coeficientes", linhas, ["Termo", "Valor"], [[t, fmt_pm(v, e)] for t, v, e in zip(termos, valores, erros)],
              figsize=(6.0, 6.8), first_col_width=0.16,
              csv=pd.DataFrame({"coeficiente": [l.replace("$", "") for l in linhas], "relacao": ["existencia"] * 10 + ["recorrencia"],
                                "termo": TERMOS_TAB + ["S(f) = 1 + s f^2"], "valor": valores, "erro_padrao": erros}))
gravar_relacoes(existencia={"coef": COEF_EXIST.tolist(), "erro": ERRO_EXIST.tolist(), "termos": TERMOS_TAB, "metricas": MET_EXIST},
                recorrencia={"forma": "S(f) = 1 + s f^2", "s": S2, "erro": ERRO_S2, "familias": N_FAM_S, "validacao": VALIDACAO, "espacamento": ESPACAMENTO},
                teste="20 % das curvas, semente 42")
print("coeficientes gravados em", ARQ_REL)

## 10. Comparação com as formas ajustadas

A Eq. (4.43) troca o passo da esfera homogênea pelo espaçamento do core-shell. Para ver o que cada ingrediente acrescenta, as transições
$x^{(\ell)}_{M,n}\to x^{(\ell)}_{M,n+1}$ das famílias de teste são previstas por quatro passos da mesma forma,
$x^{(\ell)}_{M,n+1}=x^{(\ell)}_{M,n}+\lambda\,(z_{\ell-1,n+1}-z_{\ell-1,n})$:

| Modelo | $\lambda$ | O que representa |
|---|---|---|
| Esfera $A$ | $1/(c_M/c_A)=\alpha^{1-f}$ | só o núcleo, casca ignorada ($\delta\to0$) |
| Esfera $B$ de raio $b$ | $1/[(1+\delta)\,c_M/c_B]=\beta^{1-f}/(1+\delta)$ | partícula inteira com o material da casca |
| Core-shell, $S=1$ | $\Delta_W/\pi=e^{-u}$ | fase acumulada nas duas regiões, sem ajuste |
| Core-shell, $S(f)$ | $S(f)\,e^{-u}$ | Eq. (4.43) com o $S(f)$ ajustado na Seção 8 |

As métricas são $R^2$ e MAPE de $x_{M,n+1}$ e o erro relativo do passo $x_{M,n+1}-x_{M,n}$, mostrado em função de $f$ e na distribuição acumulada.

In [ ]:
def transicoes_familia(fam):
    t = fam.sort_values(["curva", "l", "xM_peak"])
    mesmo = (t["curva"].to_numpy()[1:] == t["curva"].to_numpy()[:-1]) & (t["l"].to_numpy()[1:] == t["l"].to_numpy()[:-1])
    a, b = t.iloc[:-1].loc[mesmo], t.iloc[1:].loc[mesmo]
    tr = a[["curva", "l", "alpha", "beta", "delta", "f", "xM_peak", "z"]].rename(columns={"xM_peak": "x_n", "z": "z_n"}).reset_index(drop=True)
    tr["x_np1"], tr["z_np1"] = b["xM_peak"].to_numpy(), b["z"].to_numpy()
    mat = materiais(tr["alpha"].to_numpy(), tr["beta"].to_numpy(), tr["f"].to_numpy())
    tr["mA"], tr["mB"] = mat[3] / mat[4], mat[3] / mat[5]
    return tr


def r2_mape(obs, pred):
    ok = np.isfinite(obs) & np.isfinite(pred)
    obs, pred = obs[ok], pred[ok]
    if len(obs) < 2:
        return np.nan, np.nan
    return 1 - np.sum((obs - pred) ** 2) / np.sum((obs - obs.mean()) ** 2), 100 * np.mean(np.abs((obs - pred) / obs))


TR = transicoes_familia(FAM_TESTE)
PASSOS = {"Esfera $A$": lambda t: 1 / t["mA"], "Esfera $B$ de raio $b$": lambda t: 1 / ((1 + t["delta"]) * t["mB"]),
          "Core-shell, $S=1$": lambda t: 1 / (t["mA"] + t["delta"] * t["mB"]), "Core-shell, $S(f)$": lambda t: S_de_f(t["f"].to_numpy()) / (t["mA"] + t["delta"] * t["mB"])}
COMPARA, ERRO_PASSO = [], {}
for nome, lam in PASSOS.items():
    pred = TR["x_n"] + lam(TR) * (TR["z_np1"] - TR["z_n"])
    passo = TR["x_np1"] - TR["x_n"]
    ERRO_PASSO[nome] = (100 * np.abs(pred - TR["x_n"] - passo) / passo).to_numpy()
    R2, mape = r2_mape(TR["x_np1"].to_numpy(), pred.to_numpy())
    COMPARA.append(dict(modelo=nome.replace("$", ""), transicoes=len(TR), R2=R2, MAPE=mape, passo=float(np.nanmedian(ERRO_PASSO[nome])) if len(TR) else np.nan))
COMPARA = pd.DataFrame(COMPARA)
print(COMPARA.to_string(index=False, float_format=lambda v: f"{v:.4g}"))
COMPARA.to_csv(os.path.join(TAB_DIR, "comparacao_esfera_coreshell.csv"), index=False)

if len(TR):
    k_f, c_f = faixas_f(TR)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3.5))
    for i, nome in enumerate(PASSOS):
        e = ERRO_PASSO[nome]
        med = pd.Series(e, index=c_f[k_f]).groupby(level=0).median()
        axes[0].plot(med.index, med.to_numpy(), color=COLORS[i], linestyle=LINESTYLES[i], linewidth=LINEWIDTHS[i], marker="o", markersize=3, label=nome)
        e = np.sort(e[np.isfinite(e) & (e > 0)])
        if len(e):
            axes[1].plot(e, np.arange(1, len(e) + 1) / len(e), color=COLORS[i], linestyle=LINESTYLES[i], linewidth=LINEWIDTHS[i], label=nome)
    axes[0].set_yscale("log")
    formatar_log(axes[0])
    eixo_x_fixo(axes[0], [0.0, 0.25, 0.5, 0.75, 1.0])
    estilizar(axes[0], r"$f$", f"Erro mediano do passo ({PCT})")
    axes[1].set_xscale("log")
    formatar_log(axes[1], "x")
    axes[1].set_ylim(0, 1.02)
    axes[1].yaxis.set_major_formatter(FuncFormatter(lambda v, _: f"${dec(v, 2)}$"))
    estilizar(axes[1], f"Erro relativo do passo ({PCT})", "Fração das transições")
    adicionar_legenda(axes[1], loc="lower right", fontsize=8)
    fig.subplots_adjust(left=0.07, right=0.985, bottom=0.16, top=0.98, wspace=0.25)
    concluir(fig, "grafico_comparacao_esfera_coreshell")

## 11. Sistemas físicos: a recorrência como prova

Liu *et al.* (2000), núcleo de chumbo, casca de borracha de silicone e meio de epóxi, e Marston (2025), núcleo e meio de ar com casca densa. As linhas
cinzas são as posições dadas pela relação de recorrência, Eq. (4.43), com o primeiro termo das Eqs. (4.45)/(4.46) e o $S(f)$ ajustado na Seção 8
(sem nenhuma busca de raiz), e $P_{\mathrm{res}}$ vem da relação de existência ajustada na Seção 7. Como esses sistemas não seguem exatamente a
parametrização das grades, cada um recebe um fator de mistura efetivo, ajustado por mínimos quadrados em
$\ln(\rho_i/\rho_M)=f\,\ln(Z_i/Z_M)$, $i=A,B$. A energia mostrada é a do código original desta figura,
$2\big(\widetilde W_A/Z_A+\widetilde W_B/Z_B\big)$, em função de $y_M=(1+\delta)x_M$; os pontos são os máximos com proeminência de pelo menos $0{,}75$ em $\log_{10}$.

In [ ]:
SISTEMAS = {"liu2000": dict(rotulo="Liu et al. (2000)", mat=(1180.0, 11600.0, 1300.0, 2539.52, 2493.09, 22.8709), delta=0.42, y_lit=None,
                            yticks=[1e-7, 1e-3, 1e1, 1e5, 1e9], ylim=(1e-7, 1e10), familias=[0]),
            "marston2025": dict(rotulo="Marston (2025)", mat=(1.22, 1.22, 841.8 * 1.22, 343.0, 343.0, 4.519 * 343.0), delta=(1 - 0.7) / 0.7, y_lit=0.1554,
                                yticks=[1e-15, 1e-11, 1e-7, 1e-3, 1e1], ylim=(1e-15, 1e2), familias=[0])}
YLABEL = r"$\frac{2f_0}{\left|A\right|^2}\left(\frac{W^T_{\mathrm{n\acute{u}cleo}}}{Z_A}+\frac{W^T_{\mathrm{casca}}}{Z_B}\right)$"


def f_efetivo(mat):
    rM, rA, rB, cM, cA, cB = mat
    lz, lr = np.log([rA * cA / (rM * cM), rB * cB / (rM * cM)]), np.log([rA / rM, rB / rM])
    return float(np.clip(np.sum(lz * lr) / np.sum(lz * lz), 0, 1)) if np.sum(lz * lz) > 0 else 0.0


def energia_sistema(xM, delta, mat):
    rM, rA, rB, cM, cA, cB = mat
    WA, WB = energias_regioes(xM, delta, mat)
    return 2.0 * (WA / (rA * cA) + WB / (rB * cB))


xM_vals = np.linspace(1e-4, 0.75, 1000)
for tag, s in SISTEMAS.items():
    mat, delta = s["mat"], s["delta"]
    rM, rA, rB, cM, cA, cB = mat
    alpha, beta = rA * cA / (rM * cM), rB * cB / (rM * cM)
    W = energia_sistema(xM_vals, delta, mat)
    yM = (1 + delta) * xM_vals
    logW = np.where(np.isfinite(W) & (W > 0), np.log10(np.where(W > 0, W, 1.0)), -np.inf)
    idx, _ = find_peaks(logW, prominence=0.75)
    f_ef = f_efetivo(mat)
    linhas_rec = (1 + delta) * np.sort(np.concatenate([familia(l, delta, mat, float(S_de_f(f_ef)), x_max=xM_vals.max()) for l in s["familias"]] + [np.array([])]))
    s.update(alpha=alpha, beta=beta, f_ef=f_ef, P_res=float(p_res(alpha, beta, delta, f_ef, COEF_EXIST)[0]),
             y_pico1=float(yM[idx[0]]) if len(idx) else float(yM[np.nanargmax(W)]))
    print(f"{s['rotulo']}: alpha = {alpha:.4g}, beta = {beta:.4g}, delta = {delta:.4f}, f efetivo = {f_ef:.3f}, S = {float(S_de_f(f_ef)):.4f}, P_res = {s['P_res']:.4f}")
    print("   recorrência, y_M = " + "; ".join(f"{v:.4f}" for v in linhas_rec))
    print("   máximos de W_T, y_M = " + "; ".join(f"{v:.4f}" for v in yM[idx]))
    fig, ax = plt.subplots(figsize=(5, 3.5))
    for yp in linhas_rec:
        ax.axvline(yp, color="0.7", linestyle=":", linewidth=1.2, zorder=1)
    ax.plot(yM, W, color=COLORS[0], linestyle=LINESTYLES[0], linewidth=LINEWIDTHS[0], zorder=2)
    ax.scatter(yM[idx], W[idx], s=16, color=COLORS[0], edgecolors="white", linewidths=0.45, zorder=5)
    if s["y_lit"] is not None:
        ax.axvline(s["y_lit"], color=COLORS[1], linestyle=LINESTYLES[1], linewidth=LINEWIDTHS[1], ymin=0.0, ymax=0.55, zorder=4,
                   label=rf"$y_M^{{\mathrm{{lit.}}}}={dec(s['y_lit'], 4)}$")
        adicionar_legenda(ax)
    eixo_x_fixo(ax, [0.25, 0.5, 0.75, 1.0], (0.0, yM.max()))
    eixo_log_y(ax, s["yticks"], s["ylim"])
    estilizar(ax, r"$y_M=(1+\delta)\,x_M$", YLABEL)
    adicionar_caixa(ax, s["rotulo"] + "\n" + rf"$\alpha={gnum(alpha)},\ \beta={gnum(beta)}$" + "\n"
                    + rf"$\delta={dec(delta, 3)},\ P_{{\mathrm{{res}}}}={dec(s['P_res'], 4)}$")
    fig.subplots_adjust(left=0.18, right=0.985, bottom=0.15, top=0.98)
    concluir(fig, f"validacao_{tag}")

### Campos internos

$|\varphi|^2$ no plano $xz$, em escala logarítmica, na primeira ressonância resolvida e em $y_M=1$, para Liu *et al.* (2000) e Marston (2025). As
circunferências tracejada e contínua são as interfaces núcleo-casca e casca-meio externo.

In [ ]:
def intensidade_interna(mat, delta, xM, res=1000):
    # |phi|^2 no núcleo e na casca (plano xz, a = 1), com os coeficientes da Seção 2
    a, b = 1.0, 1.0 + delta
    rM, rA, rB, cM, cA, cB = mat
    lim = 1.15 * b
    X, Z = np.meshgrid(np.linspace(-lim, lim, res), np.linspace(-lim, lim, res))
    r = np.sqrt(X ** 2 + Z ** 2)
    cos_t = np.divide(Z, r, out=np.zeros_like(Z), where=r != 0)
    phi = np.zeros_like(r, dtype=complex)
    nuc, cas = r <= a, (r > a) & (r <= b)
    for ell in range(int(max(15, np.ceil(b * (cM / cB) * xM + 20))) + 1):
        b_l, c_l, d_l, _, _ = coeficientes(ell, xM, delta, mat)
        fator, P = (2 * ell + 1) * 1j ** ell, eval_legendre(ell, cos_t)
        phi[nuc] += fator * b_l * J(ell, (cM / cA) * xM * r[nuc]) * P[nuc]
        phi[cas] += fator * (c_l * J(ell, (cM / cB) * xM * r[cas]) + d_l * Y(ell, (cM / cB) * xM * r[cas])) * P[cas]
    I = np.maximum(np.abs(phi) ** 2, 1e-12)
    return np.ma.masked_where(r > b, I), lim


for tag, s in SISTEMAS.items():
    xs = (s["y_pico1"] / (1 + s["delta"]), 1.0 / (1 + s["delta"]))
    campos = [intensidade_interna(s["mat"], s["delta"], x) for x in xs]
    vals = np.concatenate([c[0].compressed() for c in campos])
    vals = vals[np.isfinite(vals) & (vals > 0)]
    norm = LogNorm(vmin=vals.min(), vmax=vals.max())
    fig = plt.figure(figsize=(10, 3.5))
    axes = [fig.add_axes([0.08, 0.06, 0.36, 0.84]), fig.add_axes([0.46, 0.06, 0.36, 0.84])]
    cax = fig.add_axes([0.80, 0.06, 0.015, 0.84])
    for ax, (I, lim), x in zip(axes, campos, xs):
        im = ax.imshow(I, extent=[-lim, lim, -lim, lim], origin="lower", cmap=RESONANCE_CMAP.with_extremes(bad="white"), norm=norm)
        ax.add_artist(Circle((0, 0), 1.0, color="black", fill=False, linestyle="--", linewidth=1.0))
        ax.add_artist(Circle((0, 0), 1.0 + s["delta"], color="black", fill=False, linewidth=1.2))
        ax.set_aspect("equal"); ax.set_xticks([]); ax.set_yticks([])
        estilizar(ax)
        ax.set_title("$y_M=" + virgula(f"{(1 + s['delta']) * x:.4g}") + "$", fontsize=12, pad=4)
    cbar = fig.colorbar(im, cax=cax)
    cbar.ax.yaxis.set_major_formatter(FuncFormatter(potencia))
    cbar.set_label(r"$|\varphi|^2$", fontsize=12, labelpad=16, rotation=-90)
    cbar.ax.tick_params(labelsize=10, width=0.9, length=4, direction="in")
    cbar.outline.set_edgecolor("black"); cbar.outline.set_linewidth(1.0)
    adicionar_caixa(axes[0], s["rotulo"], x=0.035, ha="left")
    print(f"{s['rotulo']}: primeira ressonância em y_M = {s['y_pico1']:.4f}; painéis em y_M = {s['y_pico1']:.4f} e 1")
    concluir(fig, f"campo_interno_{tag}")

## 12. Custo computacional

Tempo do cálculo completo por configuração (`runtime_s` do gerador, na máquina onde o banco rodou) contra o tempo das relações medido aqui:
$P_{\mathrm{res}}$ e todas as posições até $x_M=1$ (todas as ordens $\ell$), pelas Eqs. (4.43), (4.45) e (4.46), para cada configuração de teste.

In [ ]:
t0 = time.perf_counter()
for _ in range(5):
    p_res(RESUMO["alpha"].to_numpy(), RESUMO["beta"].to_numpy(), RESUMO["delta"].to_numpy(), RESUMO["f"].to_numpy(), COEF_EXIST)
t_exist = (time.perf_counter() - t0) / 5 / len(RESUMO)

CUSTO = []
for _, c in teste_c.iterrows():
    mat = materiais(c["alpha"], c["beta"], c["f"])
    S = float(S_de_f(c["f"]))
    t0 = time.perf_counter()
    n = sum(len(familia(l, c["delta"], mat, S)) for l in range(LMAX + 1))
    CUSTO.append(dict(Phi=c["Phi"], N_rel=n, relacoes=time.perf_counter() - t0 + t_exist, completo=c.get("runtime_s", np.nan)))
CUSTO = pd.DataFrame(CUSTO)
print(f"relação de existência: {t_exist * 1e6:.3g} microssegundos por configuração")
print(f"relações completas (P_res + todas as posições): mediana {CUSTO['relacoes'].median() * 1e3:.3g} ms por configuração")
if CUSTO["completo"].notna().any():
    print(f"cálculo completo do gerador: mediana {CUSTO['completo'].median():.3g} s | ganho mediano {np.nanmedian(CUSTO['completo'] / CUSTO['relacoes']):.3g}x")

fig, ax = plt.subplots(figsize=(5, 3.5))
if CUSTO["completo"].notna().any():
    ax.scatter(CUSTO["Phi"], CUSTO["completo"], s=8, facecolors="white", edgecolors="0.45", linewidths=0.5, label="Cálculo completo", zorder=2)
ax.scatter(CUSTO["Phi"], CUSTO["relacoes"], s=10, facecolors="#8B1A1A", edgecolors="black", linewidths=0.3, label="Relações", zorder=3)
ax.axhline(t_exist, color="black", linestyle=":", linewidth=1.0, label="Só a existência")
ax.set_xscale("log"); ax.set_yscale("log")
formatar_log(ax, "xy")
estilizar(ax, r"$\Phi=c_M/c_A+\delta\,c_M/c_B$", "Tempo por configuração (s)")
adicionar_legenda(ax, loc="center right")
fig.subplots_adjust(left=0.16, right=0.985, bottom=0.15, top=0.98)
concluir(fig, "grafico_custo")

## 13. Arquivos gerados

Lista o que foi gravado em `resultados/` e cria `resultados_coreshell.zip` ao lado da pasta (no Colab, dentro de `Meu Drive/coreshell_final/`),
pronto para baixar.

In [ ]:
for sub in (FIG_DIR, TAB_DIR):
    arqs = sorted(os.listdir(sub))
    nomes = sorted({os.path.splitext(a)[0] for a in arqs})
    print(f"{sub}: {len(arqs)} arquivos, {len(nomes)} itens")
    for n in nomes:
        print("   ", n)
zip_path = shutil.make_archive(os.path.join(os.path.dirname(SAIDA), "resultados_coreshell"), "zip",
                               root_dir=os.path.dirname(SAIDA), base_dir=os.path.basename(SAIDA))
print("zip:", zip_path, f"({os.path.getsize(zip_path) / 1e6:.1f} MB)")
if NO_COLAB:
    print("No Colab os arquivos já estão no Drive; a sincronização pode levar alguns minutos para aparecer no navegador.")